# Controlled Dice experiment

This notebook now uses `T=3` and adds `0.5 * Dice` to the rain-head BCE loss only. The ConvLSTM architecture, environmental preprocessing, land-use setting, intensity loss, dry loss, `beta=0.05`, and raw intensity forecast remain unchanged.

After training, run the final validation cell to sweep rain-probability thresholds, select the threshold by CSI, sweep component sizes, compare gated intensity metrics, and create the target/raw/probability/mask/gated montage.


Training windows now advance by five minutes within a whole-date training partition. Validation/test windows remain non-overlapping. Restart the kernel and rerun from the start. More windows mean longer epochs; compare runs by optimizer updates, not epoch count. The new date-based split differs from older sample-based results, so rerun a matched baseline before claiming improvement.


In [ ]:
import os
os.environ.pop("MPLBACKEND", None)

%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
import os
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
# or force it:
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA")

In [ ]:
import os
SEED = 67


import gc
import sys
from pathlib import Path

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
from datetime import datetime
from scipy import signal

print("torch:", torch.__version__)
print("cuda build:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("cuda device name:", torch.cuda.get_device_name(0))
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


torch.manual_seed(SEED)

In [ ]:
from data_processing.multimodal_radar_dataset import radar_dataset_multimodal

In [ ]:
import importlib
import json
from pathlib import Path
import data_processing.multimodal_radar_dataset as multimodal_dataset_module

importlib.reload(multimodal_dataset_module)
radar_dataset_multimodal = multimodal_dataset_module.radar_dataset_multimodal

# T=3: three input radar/environment frames predict t+5, then t+5 is rolled back into
# the same model (with persisted t environment) to predict t+10.
sequence_length = 3
num_target_steps = 2
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/data_processing").is_dir())
land_use_path = project_root / "data" / "land_use_masks.npy"
dataset = radar_dataset_multimodal(
    str(project_root / "data" / "70km" / "png"),
    str(project_root / "data" / "environment"),
    list_length=sequence_length,
    total=None,
    num_workers=14,
    land_use_path=str(land_use_path),
    use_land_use=False,
    num_target_steps=num_target_steps,
    overlapping_training=True,
)
# Whole dates are partitioned before windows; training advances five minutes.
train_indices = dataset.split_indices["train"]
val_indices = dataset.split_indices["validation"]
test_indices = dataset.split_indices["test"]
print({name: len(indices) for name, indices in dataset.split_indices.items()})
manifest_path = project_root / "models" / "overlapping_long_manifest.json"
manifest_path.write_text(json.dumps(dataset.frame_manifest, indent=2), encoding="utf-8")
dataset.fit_normalization(train_indices)
normalization_stats = dataset.get_normalization_stats()
norm_stats_path = project_root / "models" / "normalization_stats.json"
norm_stats_path.write_text(json.dumps(normalization_stats, indent=2), encoding="utf-8")
print(f"Fitted and saved training-split normalization stats: {norm_stats_path}")

# Chronological whole-date 75/15/10 split: val is used for early stopping/checkpoint selection,
# test is only ever touched once at the very end.
train_dataset = torch.utils.data.Subset(dataset, train_indices)
val_dataset = torch.utils.data.Subset(dataset, val_indices)
test_dataset = torch.utils.data.Subset(dataset, test_indices)

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE =8 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Train batches :{len(train_loader)}")
print(f"Val batches :{len(val_loader)}")
print(f"Test batches :{len(test_loader)}")

In [ ]:
for inputs, labels in train_loader:
    print("inputs shape:", inputs.shape)
    print("labels shape:", labels.shape)  # [B, num_target_steps, H, W] -> +5 and +10 radar targets
    assert inputs.shape[2] == 7, f"Expected 7 channels, got {inputs.shape[2]}"
    assert labels.shape[1] == 2, f"Expected 2 target steps (+5, +10), got {labels.shape[1]}"
    break

In [ ]:
import importlib
import models.multi_modal_convlstm as multimodal_model_module

importlib.reload(multimodal_model_module)
ConvLSTM_MM = multimodal_model_module.ConvLSTM_MM

model = ConvLSTM_MM(input_dim=7,
                   hidden_dim=[32, 64],
                   kernel_size=[(3, 3), (3, 3)],
                   num_layers=2,
                   batch_first=True,
                   bias=True,
                   use_land_use=False)

model = model.to(device)

In [ ]:
import torch.optim
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
# Define loss functions

classification_weight = 1.0
rain_intensity_weight = 1.0
dry_weight = 0.25
pos_weight = torch.tensor([20.0], device=device)
DICE_WEIGHT = 0.5


def dice_loss(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    intersection = (prob * target).sum(dim=(1, 2, 3))
    denominator = prob.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = (2.0 * intersection + eps) / (denominator + eps)

    has_rain = target.sum(dim=(1, 2, 3)) > 0
    if has_rain.any():
        return 1.0 - dice[has_rain].mean()
    return logits.new_tensor(0.0)


def dual_head_loss(
    pred,
    target,
    rain_logits,
    rain_threshold=0.01,
    classification_weight=1.0,
    rain_intensity_weight=1.0,
    dry_weight=0.25,
    beta=0.05,
    pos_weight=None,
    dice_weight=DICE_WEIGHT,
    return_components=False,
):
    rain_target = (target > rain_threshold).float()
    rain_mask = rain_target.bool()
    dry_mask = ~rain_mask

    bce_loss = F.binary_cross_entropy_with_logits(
        rain_logits,
        rain_target,
        pos_weight=pos_weight,
    )
    dice_value = dice_loss(rain_logits.unsqueeze(1), rain_target.unsqueeze(1))
    weighted_dice = dice_weight * dice_value
    rain_loss = bce_loss + weighted_dice

    if rain_mask.any():
        intensity_loss = F.smooth_l1_loss(
            pred[rain_mask],
            target[rain_mask],
            beta=beta,
        )
    else:
        intensity_loss = pred.new_tensor(0.0)

    if dry_mask.any():
        dry_loss = F.smooth_l1_loss(
            pred[dry_mask],
            torch.zeros_like(pred[dry_mask]),
            beta=beta,
        )
    else:
        dry_loss = pred.new_tensor(0.0)

    total_loss = (
        classification_weight * rain_loss
        + rain_intensity_weight * intensity_loss
        + dry_weight * dry_loss
    )

    if return_components:
        return total_loss, bce_loss, dice_value, rain_loss, intensity_loss, dry_loss
    return total_loss




In [ ]:
loss_fn = dual_head_loss

In [ ]:
LAMBDA_SECOND_STEP = 0.25


def build_rollout_inputs(inputs, pred_full):
    """Shift the input window forward, replacing the oldest frame with the model's
    own +5 radar prediction plus the latest known (persisted) environment channels.
    """
    latest_env = inputs[:, -1, 1:, :, :]
    next_frame = torch.cat([pred_full, latest_env], dim=1)
    return torch.cat([inputs[:, 1:, :, :, :], next_frame.unsqueeze(1)], dim=1)


def train_one_epoch(epoch_index, optimizer, model, loss_fn, lambda_second_step=LAMBDA_SECOND_STEP):
    running_loss = 0.0
    running_loss_plus5 = 0.0
    running_loss_plus10 = 0.0
    epoch_loss_sum = 0.0
    epoch_sample_count = 0

    for i, data in enumerate(train_loader):
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)

        # labels: [B, 2, H, W] -> target1 = t+5, target2 = t+10
        target1 = labels[:, 0]
        target2 = labels[:, 1]

        optimizer.zero_grad()

        # --- +5 minute forecast ---
        pred1_full, _, _, rain_logits1_full = model(inputs, return_logits=True)
        pred1 = pred1_full.squeeze(1)
        rain_logits1 = rain_logits1_full.squeeze(1)

        loss1, bce_loss1, dice_value1, rain_loss1, intensity_loss1, dry_loss1 = loss_fn(
            pred1,
            target1,
            rain_logits1,
            return_components=True,
        )

        # --- +10 minute forecast: feed pred1 back in (no detach, so gradients flow
        # through the first pass), keeping the environment channels persisted from t.
        next_inputs = build_rollout_inputs(inputs, pred1_full)
        pred2_full, _, _, rain_logits2_full = model(next_inputs, return_logits=True)
        pred2 = pred2_full.squeeze(1)
        rain_logits2 = rain_logits2_full.squeeze(1)

        loss2, bce_loss2, dice_value2, rain_loss2, intensity_loss2, dry_loss2 = loss_fn(
            pred2,
            target2,
            rain_logits2,
            return_components=True,
        )

        total_loss = loss1 + lambda_second_step * loss2

        total_loss.backward()
        optimizer.step()

        # Include every batch, weighting the final partial batch by its size.
        batch_size = inputs.size(0)
        epoch_loss_sum += total_loss.item() * batch_size
        epoch_sample_count += batch_size
        running_loss += total_loss.item()
        running_loss_plus5 += loss1.item()
        running_loss_plus10 += loss2.item()

        if i % 10 == 9:
            window_loss = running_loss / 10
            print(f"  batch {i + 1} total={window_loss:.4f}")
            print(
                f"  +5  BCE={bce_loss1.item():.4f} Dice={dice_value1.item():.4f} "
                f"rain={rain_loss1.item():.4f} intensity={intensity_loss1.item():.4f} "
                f"dry={dry_loss1.item():.4f} loss={running_loss_plus5 / 10:.4f}"
            )
            print(
                f"  +10 BCE={bce_loss2.item():.4f} Dice={dice_value2.item():.4f} "
                f"rain={rain_loss2.item():.4f} intensity={intensity_loss2.item():.4f} "
                f"dry={dry_loss2.item():.4f} loss={running_loss_plus10 / 10:.4f} "
                f"(weighted={lambda_second_step * running_loss_plus10 / 10:.4f})"
            )
            running_loss = 0.0
            running_loss_plus5 = 0.0
            running_loss_plus10 = 0.0

    return epoch_loss_sum / max(epoch_sample_count, 1)

In [ ]:
from data_processing.model_contract import write_contract
epoch_number = 0
val = []
train = []
val_plus10 = []
EPOCHS = 400
EARLY_STOPPING_PATIENCE = 80

best_vloss = float("inf")
epochs_without_improvement = 0
model_path = project_root / "models" / "model_best_long.pkl"

for epoch in range(EPOCHS):
    print(f"EPOCH {epoch + 1}:")

    model.train(True)
    avg_train_loss = train_one_epoch(epoch_number, optimizer, model, loss_fn)

    running_vloss_plus5 = 0.0
    running_vloss_plus10 = 0.0
    validation_components = {"bce": 0.0, "dice": 0.0, "rain": 0.0, "intensity": 0.0, "dry": 0.0}
    validation_components_plus10 = {"bce": 0.0, "dice": 0.0, "rain": 0.0, "intensity": 0.0, "dry": 0.0}
    model.eval()

    with torch.no_grad():
        for validation_batch_index, vdata in enumerate(val_loader):
            vinputs, vlabels = vdata
            vinputs = vinputs.to(device)
            vlabels = vlabels.to(device)

            vtarget1 = vlabels[:, 0]
            vtarget2 = vlabels[:, 1]

            # +5 forecast
            voutputs1_full, _, _, v_rain_logits1_full = model(vinputs, return_logits=True)
            voutputs1 = voutputs1_full.squeeze(1)
            v_rain_logits1 = v_rain_logits1_full.squeeze(1)
            vloss1, vbce1, vdice1, vrain1, vintensity1, vdry1 = loss_fn(
                voutputs1, vtarget1.float(), v_rain_logits1, return_components=True
            )

            # +10 forecast: same rollout used in training (pred1 fed back in, never vtarget1)
            v_next_inputs = build_rollout_inputs(vinputs, voutputs1_full)
            voutputs2_full, _, _, v_rain_logits2_full = model(v_next_inputs, return_logits=True)
            voutputs2 = voutputs2_full.squeeze(1)
            v_rain_logits2 = v_rain_logits2_full.squeeze(1)
            vloss2, vbce2, vdice2, vrain2, vintensity2, vdry2 = loss_fn(
                voutputs2, vtarget2.float(), v_rain_logits2, return_components=True
            )

            running_vloss_plus5 += vloss1.item()
            running_vloss_plus10 += vloss2.item()

            validation_components["bce"] += vbce1.item()
            validation_components["dice"] += vdice1.item()
            validation_components["rain"] += vrain1.item()
            validation_components["intensity"] += vintensity1.item()
            validation_components["dry"] += vdry1.item()

            validation_components_plus10["bce"] += vbce2.item()
            validation_components_plus10["dice"] += vdice2.item()
            validation_components_plus10["rain"] += vrain2.item()
            validation_components_plus10["intensity"] += vintensity2.item()
            validation_components_plus10["dry"] += vdry2.item()

    number_validation_batches = max(len(val_loader), 1)
    avg_val_loss_plus5 = running_vloss_plus5 / number_validation_batches
    avg_val_loss_plus10 = running_vloss_plus10 / number_validation_batches
    validation_components = {name: value / number_validation_batches for name, value in validation_components.items()}
    validation_components_plus10 = {
        name: value / number_validation_batches for name, value in validation_components_plus10.items()
    }
    print(
        f"  +5  validation total={avg_val_loss_plus5:.4f} "
        f"BCE={validation_components['bce']:.4f} Dice={validation_components['dice']:.4f} "
        f"rain total={validation_components['rain']:.4f} intensity={validation_components['intensity']:.4f} "
        f"dry={validation_components['dry']:.4f}"
    )
    print(
        f"  +10 validation total={avg_val_loss_plus10:.4f} "
        f"BCE={validation_components_plus10['bce']:.4f} Dice={validation_components_plus10['dice']:.4f} "
        f"rain total={validation_components_plus10['rain']:.4f} intensity={validation_components_plus10['intensity']:.4f} "
        f"dry={validation_components_plus10['dry']:.4f}"
    )

    # Probe metrics use the +5 forecast only - that's the primary research question.
    with torch.no_grad():
        probe_inputs, probe_labels = next(iter(val_loader))
        probe_inputs = probe_inputs.to(device)
        probe_target1 = probe_labels[:, 0].to(device)
        probe_outputs_full, _, probe_delta, probe_rain_logits_full = model(probe_inputs, return_logits=True)
        probe_outputs = probe_outputs_full.squeeze(1)
        probe_delta = probe_delta.squeeze(1)
        probe_rain_logits = probe_rain_logits_full.squeeze(1)
        rain_probs = torch.sigmoid(probe_rain_logits)
        probe_threshold = 0.40
        pred_rain = rain_probs >= probe_threshold
        target_rain = probe_target1 > 0.01
        pred_rate = pred_rain.float().mean().item()
        target_rate = target_rain.float().mean().item()
        precision = (pred_rain & target_rain).float().sum().item() / pred_rain.float().sum().clamp_min(1).item()
        recall = (pred_rain & target_rain).float().sum().item() / target_rain.float().sum().clamp_min(1).item()
        csi = (pred_rain & target_rain).float().sum().item() / (pred_rain | target_rain).float().sum().clamp_min(1).item()
        rainy_mask = target_rain.bool()
        dry_mask = ~rainy_mask
        rainy_pixel_mae = torch.abs(probe_outputs[rainy_mask] - probe_target1[rainy_mask]).mean().item() if rainy_mask.any() else 0.0
        dry_pixel_mae = torch.abs(probe_outputs[dry_mask] - probe_target1[dry_mask]).mean().item() if dry_mask.any() else 0.0
        dry_pred_gt_001 = (probe_outputs[dry_mask] > 0.01).float().mean().item() if dry_mask.any() else 0.0
        dry_pred_gt_005 = (probe_outputs[dry_mask] > 0.05).float().mean().item() if dry_mask.any() else 0.0
        print(
            f"  +5 probe @ threshold={probe_threshold:.2f}: target={target_rate:.3f}, pred={pred_rate:.3f}, "
            f"precision={precision:.3f}, recall={recall:.3f}, csi={csi:.3f}, "
            f"rainy_mae={rainy_pixel_mae:.4f}, dry_mae={dry_pixel_mae:.4f}, "
            f"dry_pred>0.01={dry_pred_gt_001:.3f}, dry_pred>0.05={dry_pred_gt_005:.3f}"
        )

    print(f"LOSS train {avg_train_loss} valid(+5) {avg_val_loss_plus5} valid(+10) {avg_val_loss_plus10}")
    val.append(avg_val_loss_plus5)
    train.append(avg_train_loss)
    val_plus10.append(avg_val_loss_plus10)

    # Model selection is driven by the +5 forecast only, matching the primary research
    # question (does the +10 auxiliary objective improve the +5 forecast?).
    if avg_val_loss_plus5 < best_vloss:
        best_vloss = avg_val_loss_plus5
        epochs_without_improvement = 0
        torch.save(model.state_dict(), model_path)
        write_contract(model_path, project_root / "models" / "normalization_stats.json", getattr(dataset, "decoder_version", "legacy_nearest30_v1"))
        print(f"Overwrote production model checkpoint: {model_path}")
    else:
        epochs_without_improvement += 1
        print(f"Validation loss did not improve; epochs without improvement = {epochs_without_improvement}")

    epoch_number += 1
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping triggered after {epoch_number} epochs "
            f"with no validation improvement for {EARLY_STOPPING_PATIENCE} epochs."
        )
        break

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Restored best model weights from {model_path}")

# Test set is touched exactly once, here, after the checkpoint is fixed by validation.
test_loss_plus5_total = 0.0
test_loss_plus10_total = 0.0
model.eval()
with torch.no_grad():
    for tinputs, tlabels in test_loader:
        tinputs = tinputs.to(device)
        tlabels = tlabels.to(device)
        ttarget1 = tlabels[:, 0]
        ttarget2 = tlabels[:, 1]

        toutputs1_full, _, _, t_rain_logits1_full = model(tinputs, return_logits=True)
        toutputs1 = toutputs1_full.squeeze(1)
        t_rain_logits1 = t_rain_logits1_full.squeeze(1)
        tloss1 = loss_fn(toutputs1, ttarget1.float(), t_rain_logits1)

        t_next_inputs = build_rollout_inputs(tinputs, toutputs1_full)
        toutputs2_full, _, _, t_rain_logits2_full = model(t_next_inputs, return_logits=True)
        toutputs2 = toutputs2_full.squeeze(1)
        t_rain_logits2 = t_rain_logits2_full.squeeze(1)
        tloss2 = loss_fn(toutputs2, ttarget2.float(), t_rain_logits2)

        test_loss_plus5_total += tloss1.item()
        test_loss_plus10_total += tloss2.item()

number_test_batches = max(len(test_loader), 1)
print(
    f"FINAL TEST (best checkpoint, evaluated once): "
    f"+5={test_loss_plus5_total / number_test_batches:.4f} "
    f"+10={test_loss_plus10_total / number_test_batches:.4f}"
)

In [ ]:
model2 = ConvLSTM_MM(input_dim=7,
                    hidden_dim=[32, 64],
                    kernel_size=[(3, 3), (3, 3)],
                    num_layers=2,
                    batch_first=True,
                    bias=True,
                    use_land_use=False)

model2 = model2.to(device)
model2.load_state_dict(torch.load(project_root / "models" / "model_best_long.pkl", map_location=device))
print("Loaded production model_best_long.pkl into model2")

In [ ]:
plt.plot(train, label="Training Loss")
plt.plot(val, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy import ndimage


def forecast_both_horizons(inputs):
    """Use the loaded best checkpoint and the same raw rollout as training."""
    model2.eval()
    inputs = inputs.to(device)
    with torch.no_grad():
        raw5, _, _, logits5 = model2(inputs, return_logits=True)
        rollout_inputs = build_rollout_inputs(inputs, raw5)
        raw10, _, _, logits10 = model2(rollout_inputs, return_logits=True)
    raw = torch.cat([raw5, raw10], dim=1).cpu().numpy()
    probability = torch.cat([logits5.sigmoid(), logits10.sigmoid()], dim=1).cpu().numpy()
    return raw, probability


def plot_both_horizons(inputs, targets, title, gated=False):
    """Rows are +5/+10; all intensity panels share one scale per example."""
    raw, probability = forecast_both_horizons(inputs)
    target = targets.cpu().numpy()
    assert raw.shape == target.shape == probability.shape
    assert raw.shape[1] == 2
    # Apply the selected +5 calibration to both leads for visual comparison.
    # This does not claim that these settings are optimal for +10.
    threshold = float(globals().get('selected_threshold', 0.40))
    min_size = int(globals().get('selected_component_size', 25))
    calibration = '+5 validation settings' if 'selected_threshold' in globals() else 'baseline settings'
    for sample in range(len(raw)):
        vmin = min(float(raw[sample].min()), float(target[sample].min()), 0.0)
        vmax = max(float(raw[sample].max()), float(target[sample].max()), 0.01)
        error_limit = max(float(np.abs(raw[sample] - target[sample]).max()), 1e-6)
        columns = ['Target', 'Raw intensity', 'Rain probability', 'Cleaned mask', 'Gated intensity'] if gated else ['Target', 'Raw intensity', 'Prediction - target']
        fig, axes = plt.subplots(2, len(columns), figsize=(4 * len(columns), 7), squeeze=False, constrained_layout=True)
        for lead, minutes in enumerate((5, 10)):
            if gated:
                labels, _ = ndimage.label(probability[sample, lead] >= threshold, structure=np.ones((3, 3), dtype=np.uint8))
                keep = np.bincount(labels.ravel()) >= min_size
                keep[0] = False
                mask = keep[labels]
                images = [target[sample, lead], raw[sample, lead], probability[sample, lead], mask, np.maximum(raw[sample, lead], 0) * mask]
            else:
                images = [target[sample, lead], raw[sample, lead], raw[sample, lead] - target[sample, lead]]
            for col, (axis, panel, label) in enumerate(zip(axes[lead], images, columns)):
                if gated and col in (2, 3):
                    im = axis.imshow(panel, cmap='magma' if col == 2 else 'gray', vmin=0, vmax=1)
                elif not gated and col == 2:
                    im = axis.imshow(panel, cmap='coolwarm', vmin=-error_limit, vmax=error_limit)
                else:
                    im = axis.imshow(panel, cmap='viridis', vmin=vmin, vmax=vmax)
                axis.set_title(f'+{minutes} min | {label}')
                axis.set_xticks([])
                axis.set_yticks([])
                fig.colorbar(im, ax=axis, fraction=0.046, pad=0.04)
        settings = f' | {calibration}: p >= {threshold:.3f}, size >= {min_size} (both leads)' if gated else ''
        fig.suptitle(f'{title} | sample {sample}{settings}')
        plt.show()


def plot_validation_examples(max_examples=2, strongest=False):
    # Select by +5 rain coverage so the same examples can be compared at +10.
    if not len(val_dataset):
        raise RuntimeError('No validation samples available')
    candidates = []
    for idx in range(len(val_dataset)):
        _, target = val_dataset[idx]
        candidates.append((int((target[0] > 0.01).sum()), idx))
    candidates.sort()
    if strongest:
        selected = [candidates[-1]]
    else:
        # Spread examples across rain coverage, including dry conditions.
        positions = np.linspace(0, len(candidates) - 1, min(max_examples, len(candidates)), dtype=int)
        selected = [candidates[position] for position in positions]
    for count, idx in selected:
        inputs, targets = val_dataset[idx]
        plot_both_horizons(inputs.unsqueeze(0), targets.unsqueeze(0), f'Validation index {idx} | +5 rainy pixels: {count}', gated=True)


inputs, labels = next(iter(train_loader))
sample_no = min(6, len(inputs) - 1)
plot_both_horizons(inputs[sample_no:sample_no + 1], labels[sample_no:sample_no + 1], 'Training example')

In [ ]:
# Run the visualization helpers cell above first.
inputs, labels = next(iter(val_loader))
sample_no = min(6, len(inputs) - 1)
plot_both_horizons(inputs[sample_no:sample_no + 1], labels[sample_no:sample_no + 1], 'Validation example')

In [ ]:
sample_idx = 100

In [ ]:
# Input context followed by both forecast horizons from the best checkpoint.
sample_inputs, sample_target = val_dataset[sample_idx]
num_steps = sample_inputs.shape[0]
fig, axes = plt.subplots(1, num_steps, figsize=(4 * num_steps, 4), squeeze=False, constrained_layout=True)
radar = sample_inputs[:, 0].cpu().numpy()
vmax = max(float(radar.max()), 0.01)
for step, axis in enumerate(axes[0]):
    minutes_ago = 5 * (num_steps - 1 - step)
    im = axis.imshow(radar[step], cmap='viridis', vmin=0, vmax=vmax)
    axis.set_title('Input t' if minutes_ago == 0 else f'Input t-{minutes_ago} min')
    axis.set_xticks([])
    axis.set_yticks([])
    fig.colorbar(im, ax=axis)
fig.suptitle(f'Validation input context | index {sample_idx}')
plt.show()
plot_both_horizons(sample_inputs.unsqueeze(0), sample_target.unsqueeze(0), f'Validation index {sample_idx}')

In [ ]:
from data_processing.model_contract import write_contract
import json
from pathlib import Path

save_dir = project_root / "models"
save_dir.mkdir(parents=True, exist_ok=True)

model_path = save_dir / "model_long.pkl"
norm_stats_path = save_dir / "normalization_stats_long.json"
norm_stats_path.write_text(json.dumps(normalization_stats, indent=2), encoding="utf-8")
torch.save(model2.state_dict(), model_path)
write_contract(model_path, norm_stats_path, getattr(dataset, "decoder_version", "legacy_nearest30_v1"))
print(f"Saved long-rollout model: {model_path}")
print(f"Saved matching normalization stats: {norm_stats_path}")

In [ ]:
# Both forecast horizons for low/high rain-coverage validation examples.
plot_validation_examples(max_examples=2)

In [ ]:
# Both forecast horizons across ten validation rain-coverage examples.
plot_validation_examples(max_examples=10)

In [ ]:
# Both forecast horizons for the largest +5 rainy area in validation.
plot_validation_examples(strongest=True)

In [ ]:
# Both-horizon pipeline; rerun after calibration to use its selected settings.
plot_validation_examples(max_examples=2)

In [ ]:
# Calibration/strategy diagnostics below evaluate +5 minutes only.
# Use plot_validation_examples() above for paired +5/+10 forecast displays.
# Post-training validation: recalibrate the rain threshold and component cleanup.
# Run this after the controlled Dice training cell has restored the best checkpoint.
from scipy import ndimage

DICE_VALIDATION_THRESHOLDS = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60]
COMPONENT_SIZE_CANDIDATES = [10, 15, 20, 25, 30, 40]

model2.eval()
validation_probabilities = []
validation_targets = []
validation_raw_predictions = []

with torch.no_grad():
    for validation_inputs, validation_labels in val_loader:
        validation_labels = validation_labels[:, 0]  # Evaluate the primary +5-minute forecast.
        validation_inputs = validation_inputs.to(device)
        validation_labels = validation_labels.to(device)
        raw_outputs, _, _, validation_logits = model2(
            validation_inputs,
            return_logits=True,
        )
        validation_probabilities.append(torch.sigmoid(validation_logits).squeeze(1).cpu())
        validation_targets.append(validation_labels.cpu())
        validation_raw_predictions.append(raw_outputs.squeeze(1).clamp_min(0).cpu())

validation_probabilities = torch.cat(validation_probabilities)
validation_targets = torch.cat(validation_targets)
validation_raw_predictions = torch.cat(validation_raw_predictions)
validation_rain_target = validation_targets > 0.01

rain_values = validation_probabilities[validation_rain_target]
dry_values = validation_probabilities[~validation_rain_target]
print("VALIDATION RAIN PROBABILITY DISTRIBUTIONS")
print(f"true-rain mean={rain_values.mean().item():.6f}, median={rain_values.median().item():.6f}, max={rain_values.max().item():.6f}")
print(f"true-dry mean={dry_values.mean().item():.6f}, median={dry_values.median().item():.6f}, max={dry_values.max().item():.6f}")

threshold_results = []
print("\nVALIDATION THRESHOLD SWEEP")
print("threshold  precision  recall  CSI       F1")
for probability_threshold in DICE_VALIDATION_THRESHOLDS:
    predicted_rain = validation_probabilities >= probability_threshold
    true_positive = (predicted_rain & validation_rain_target).sum().item()
    false_positive = (predicted_rain & ~validation_rain_target).sum().item()
    false_negative = (~predicted_rain & validation_rain_target).sum().item()
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    csi = true_positive / max(true_positive + false_positive + false_negative, 1)
    f1 = 2.0 * precision * recall / max(precision + recall, 1e-12)
    result = {"threshold": probability_threshold, "precision": precision, "recall": recall, "csi": csi, "f1": f1}
    threshold_results.append(result)
    print(f"{probability_threshold:>9.2f}  {precision:>9.4f}  {recall:>6.4f}  {csi:>7.4f}  {f1:>7.4f}")

selected_threshold = max(threshold_results, key=lambda item: (item["csi"], item["f1"]))["threshold"]
print(f"\nSelected validation probability threshold: {selected_threshold:.2f}")


def remove_small_components(mask, minimum_size):
    cleaned = []
    structure = np.ones((3, 3), dtype=np.uint8)
    for sample_mask in mask.numpy():
        labels, _ = ndimage.label(sample_mask, structure=structure)
        sizes = np.bincount(labels.ravel())
        keep = sizes >= minimum_size
        keep[0] = False
        cleaned.append(torch.from_numpy(keep[labels]))
    return torch.stack(cleaned)


def calculate_gated_metrics(mask):
    target_rain = validation_rain_target
    true_positive = (mask & target_rain).sum().item()
    false_positive = (mask & ~target_rain).sum().item()
    false_negative = (~mask & target_rain).sum().item()
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    csi = true_positive / max(true_positive + false_positive + false_negative, 1)
    f1 = 2.0 * precision * recall / max(precision + recall, 1e-12)
    gated = validation_raw_predictions * mask.float()
    return {
        "precision": precision,
        "recall": recall,
        "csi": csi,
        "f1": f1,
        "rainy_mae": (gated[target_rain] - validation_targets[target_rain]).abs().mean().item(),
        "dry_mae": (gated[~target_rain] - validation_targets[~target_rain]).abs().mean().item(),
        "dry_gt_001": (gated[~target_rain] > 0.01).float().mean().item(),
        "dry_gt_005": (gated[~target_rain] > 0.05).float().mean().item(),
    }


print("\nVALIDATION COMPONENT-SIZE SWEEP")
print("size  precision  recall  CSI       F1       rainy_MAE  dry_MAE   dry>.01   dry>.05")
component_results = []
base_mask = validation_probabilities >= selected_threshold
for minimum_size in COMPONENT_SIZE_CANDIDATES:
    cleaned_mask = remove_small_components(base_mask, minimum_size)
    result = calculate_gated_metrics(cleaned_mask)
    result["minimum_size"] = minimum_size
    component_results.append(result)
    print(
        f"{minimum_size:>4}  {result['precision']:.4f}     {result['recall']:.4f}  "
        f"{result['csi']:.4f}   {result['f1']:.4f}   {result['rainy_mae']:.4f}     "
        f"{result['dry_mae']:.4f}    {result['dry_gt_001']:.4f}    {result['dry_gt_005']:.4f}"
    )

selected_component_size = max(component_results, key=lambda item: (item["csi"], item["f1"]))["minimum_size"]
final_validation_mask = remove_small_components(base_mask, selected_component_size)
print(f"\nSelected validation component size: {selected_component_size}")

# Report true-rain component detection by size for the selected mask.
size_bins = [("1-2", 1, 2), ("3-5", 3, 5), ("6-20", 6, 20), ("21-100", 21, 100), (">100", 101, None)]
print("TRUE RAIN COMPONENT DETECTION")
for name, lower, upper in size_bins:
    detected = total = 0
    for sample_mask, target in zip(final_validation_mask.numpy(), validation_targets.numpy()):
        labels, count = ndimage.label(target > 0.01, structure=np.ones((3, 3), dtype=np.uint8))
        for component_id in range(1, count + 1):
            component = labels == component_id
            component_size = int(component.sum())
            in_bin = component_size >= lower and (upper is None or component_size <= upper)
            if in_bin:
                total += 1
                detected += int(np.any(sample_mask & component))
    print(f"{name}: {detected}/{total} ({detected / max(total, 1):.4f})")

# Light/heavy examples: target | raw | probability | final mask | gated forecast.
example_indices = []
for index, target in enumerate(validation_targets):
    rain_pixels = int((target > 0.01).sum().item())
    if rain_pixels <= 500 and not any(item < 0 for item in example_indices):
        example_indices.append(-index - 1)
    if rain_pixels > 500 and not any(item > 0 for item in example_indices):
        example_indices.append(index)
    if len(example_indices) == 2:
        break

figure, axes = plt.subplots(len(example_indices), 5, figsize=(22, 4 * len(example_indices)), squeeze=False)
plot_titles = ["Target", "Raw prediction", "Rain probability", "Final mask", "Gated prediction"]
for row, encoded_index in enumerate(example_indices):
    index = abs(encoded_index) - 1 if encoded_index < 0 else encoded_index
    images = [
        validation_targets[index].numpy(),
        validation_raw_predictions[index].numpy(),
        validation_probabilities[index].numpy(),
        final_validation_mask[index].numpy(),
        (validation_raw_predictions[index] * final_validation_mask[index]).numpy(),
    ]
    for column, (axis, image, title) in enumerate(zip(axes[row], images, plot_titles)):
        if column == 2:
            image_plot = axis.imshow(image, cmap="magma", vmin=0.0, vmax=1.0)
            colorbar_label = "Rain probability"
        elif column == 3:
            image_plot = axis.imshow(image, cmap="gray", vmin=0.0, vmax=1.0)
            colorbar_label = "Rain mask"
        else:
            image_plot = axis.imshow(image, cmap="viridis", vmin=0.0, vmax=1.0)
            colorbar_label = "Rain intensity"
        axis.set_title(title)
        axis.axis("off")
        figure.colorbar(image_plot, ax=axis, fraction=0.046, pad=0.04, label=colorbar_label)
figure.suptitle(f"Dice model validation: threshold={selected_threshold:.2f}, component size={selected_component_size}")
figure.tight_layout()
figure.savefig("dice_model_validation_gated_montage.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# FULL VALIDATION METRICS: +5 and +10 minutes (best checkpoint)
# Run the model2 checkpoint-loading cell and +5 calibration cell first.
# Both leads use the same +5-selected mask settings for a fair comparison;
# +10 is not independently calibrated here. No test data is used.
from scipy import ndimage

if 'selected_threshold' not in globals() or 'selected_component_size' not in globals():
    raise RuntimeError('Run the +5 validation calibration cell first.')


def horizon_clean_mask(probability, threshold, minimum_size):
    cleaned = []
    for sample in probability.numpy():
        components, _ = ndimage.label(sample >= threshold, structure=np.ones((3, 3)))
        keep = np.bincount(components.ravel()) >= minimum_size
        keep[0] = False
        cleaned.append(torch.from_numpy(keep[components]))
    return torch.stack(cleaned)


def horizon_accumulate(stats, mask, prediction, target):
    rain = target > 0.01
    dry = ~rain
    stats['tp'] += (mask & rain).sum().item()
    stats['fp'] += (mask & dry).sum().item()
    stats['fn'] += (~mask & rain).sum().item()
    stats['rain_n'] += rain.sum().item()
    stats['dry_n'] += dry.sum().item()
    error = (prediction - target).abs()
    stats['rain_error'] += error[rain].sum().item()
    stats['dry_error'] += error[dry].sum().item()
    stats['dry_wet'] += (prediction[dry] > 0.01).sum().item()


def horizon_ratio(numerator, denominator):
    return numerator / denominator if denominator else float('nan')


horizon_totals = {
    (lead, method): dict.fromkeys(
        ['tp', 'fp', 'fn', 'rain_n', 'dry_n', 'rain_error', 'dry_error', 'dry_wet'], 0
    )
    for lead in (5, 10)
    for method in ('Model mask', 'Model intensity > .01', 'Persistence')
}
model2.eval()
validation_sample_count = 0
with torch.no_grad():
    for metric_inputs, metric_targets in val_loader:
        if metric_targets.ndim != 4 or metric_targets.shape[1] != 2:
            raise ValueError('Expected targets [batch, 2, height, width].')
        metric_inputs = metric_inputs.to(device)
        raw5, _, _, logits5 = model2(metric_inputs, return_logits=True)
        # Raw, ungated +5 goes into the +10 rollout, exactly as in training.
        raw10, _, _, logits10 = model2(
            build_rollout_inputs(metric_inputs, raw5), return_logits=True
        )
        persistence = metric_inputs[:, -1, 0].cpu().clamp_min(0)
        for index, (lead, raw, logits) in enumerate(((5, raw5, logits5), (10, raw10, logits10))):
            target = metric_targets[:, index].cpu()
            probability = logits.sigmoid().squeeze(1).cpu()
            raw = raw.squeeze(1).cpu().clamp_min(0)
            mask = horizon_clean_mask(probability, selected_threshold, selected_component_size)
            gated = raw * mask
            horizon_accumulate(horizon_totals[lead, 'Model mask'], mask, gated, target)
            horizon_accumulate(horizon_totals[lead, 'Model intensity > .01'], gated > 0.01, gated, target)
            horizon_accumulate(horizon_totals[lead, 'Persistence'], persistence > 0.01, persistence, target)
        validation_sample_count += len(metric_targets)

if not validation_sample_count:
    raise RuntimeError('Validation loader is empty.')

horizon_validation_results = []
for (lead, method), stats in horizon_totals.items():
    tp, fp, fn = (stats[key] for key in ('tp', 'fp', 'fn'))
    horizon_validation_results.append({
        'lead_min': lead, 'method': method,
        'precision': horizon_ratio(tp, tp + fp),
        'recall': horizon_ratio(tp, tp + fn),
        'CSI': horizon_ratio(tp, tp + fp + fn),
        'F1': horizon_ratio(2 * tp, 2 * tp + fp + fn),
        'rainy_MAE': horizon_ratio(stats['rain_error'], stats['rain_n']),
        'dry_MAE': horizon_ratio(stats['dry_error'], stats['dry_n']),
        'dry_gt_001': horizon_ratio(stats['dry_wet'], stats['dry_n']),
    })
print(f'FULL VALIDATION: {validation_sample_count} samples | loaded best checkpoint (model2)')
print(f'Both leads: probability >= {selected_threshold:.2f}, component size >= {selected_component_size}')
print('Pixel-pooled metrics; MAE uses normalized radar intensity, not mm/hr.')
print('Model mask scores the cleaned probability mask; Model intensity scores the final gated radar.')
print('Persistence repeats the last preprocessed input radar at both horizons.')
print(pd.DataFrame(horizon_validation_results).to_string(index=False, float_format=lambda x: f'{x:.4f}'))


In [ ]:
# JOINT CALIBRATION: separate +5/+10 validation sweeps
# Run the best-checkpoint loading cell first. This cell does not use test data,
# change bot settings, or overwrite the earlier +5 calibration variables.
# One inference pass per validation batch; component labeling is reused across sizes.
import numpy as np
import pandas as pd
import torch
from scipy import ndimage

JOINT_THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
JOINT_MIN_SIZES = [0, 5, 10, 20, 25, 40, 60, 80, 120]  # 0 = no filtering
JOINT_BINS = [('1-5', 1, 5), ('6-20', 6, 20), ('21-100', 21, 100), ('>100', 101, np.inf)]
joint_structure = np.ones((3, 3), dtype=np.uint8)
joint_totals = {
    (lead, threshold, size): np.zeros(8 + len(JOINT_BINS), dtype=np.float64)
    for lead in (5, 10) for threshold in JOINT_THRESHOLDS for size in JOINT_MIN_SIZES
}
joint_region_counts = {lead: np.zeros(len(JOINT_BINS), dtype=np.int64) for lead in (5, 10)}
joint_samples = 0
model2.eval()
with torch.no_grad():
    for batch_number, (joint_inputs, joint_targets) in enumerate(val_loader, 1):
        if joint_targets.ndim != 4 or joint_targets.shape[1] != 2:
            raise ValueError('Expected two-horizon targets [B, 2, H, W].')
        joint_inputs = joint_inputs.to(device)
        joint_raw5, _, _, joint_logits5 = model2(joint_inputs, return_logits=True)
        joint_raw10, _, _, joint_logits10 = model2(
            build_rollout_inputs(joint_inputs, joint_raw5), return_logits=True
        )
        for lead_index, (lead, raw_tensor, logit_tensor) in enumerate(
            ((5, joint_raw5, joint_logits5), (10, joint_raw10, joint_logits10))
        ):
            raw_batch = raw_tensor.squeeze(1).clamp_min(0).cpu().numpy()
            prob_batch = logit_tensor.squeeze(1).sigmoid().cpu().numpy()
            targets_batch = joint_targets[:, lead_index].cpu().numpy()
            for raw, probability, target in zip(raw_batch, prob_batch, targets_batch):
                if not all(np.isfinite(a).all() for a in (raw, probability, target)):
                    raise ValueError('Non-finite validation predictions or targets.')
                rain = target > 0.01
                target_labels, _ = ndimage.label(rain, structure=joint_structure)
                target_sizes = np.bincount(target_labels.ravel())
                region_bins = []
                for bin_index, (_, lower, upper) in enumerate(JOINT_BINS):
                    ids = np.flatnonzero((target_sizes >= lower) & (target_sizes <= upper))
                    ids = ids[ids != 0]
                    region_bins.append(ids)
                    joint_region_counts[lead][bin_index] += len(ids)
                for threshold in JOINT_THRESHOLDS:
                    labels, _ = ndimage.label(probability >= threshold, structure=joint_structure)
                    sizes = np.bincount(labels.ravel())
                    pixel_sizes = sizes[labels]
                    pixel_sizes[labels == 0] = 0
                    # Largest predicted component overlapping each true region.
                    # A region is detected if at least one pixel survives filtering.
                    region_support = np.zeros(len(target_sizes), dtype=np.int64)
                    np.maximum.at(region_support, target_labels.ravel(), pixel_sizes.ravel())
                    for minimum_size in JOINT_MIN_SIZES:
                        effective_size = max(1, minimum_size)
                        mask = pixel_sizes >= effective_size
                        gated = raw * mask
                        error = np.abs(gated - target)
                        stats = joint_totals[lead, threshold, minimum_size]
                        stats[:8] += [
                            np.count_nonzero(mask & rain), np.count_nonzero(mask & ~rain),
                            np.count_nonzero(~mask & rain), np.count_nonzero(rain),
                            np.count_nonzero(~rain), error[rain].sum(dtype=np.float64),
                            error[~rain].sum(dtype=np.float64), np.count_nonzero((gated > 0.01) & ~rain),
                        ]
                        for bin_index, ids in enumerate(region_bins):
                            stats[8 + bin_index] += np.count_nonzero(region_support[ids] >= effective_size)
        joint_samples += len(joint_targets)
        if batch_number % 10 == 0:
            print(f'Joint calibration: {batch_number}/{len(val_loader)} batches, {joint_samples} samples', flush=True)

if not joint_samples:
    raise RuntimeError('Validation loader is empty.')

def joint_ratio(a, b):
    return a / b if b else float('nan')

joint_rows = []
for (lead, threshold, size), stats in joint_totals.items():
    tp, fp, fn, rain_n, dry_n, rain_error, dry_error, dry_wet = stats[:8]
    row = dict(lead_min=lead, threshold=threshold, min_size=size,
               precision=joint_ratio(tp, tp + fp), recall=joint_ratio(tp, tp + fn),
               CSI=joint_ratio(tp, tp + fp + fn), F1=joint_ratio(2 * tp, 2 * tp + fp + fn),
               rainy_MAE=joint_ratio(rain_error, rain_n), dry_MAE=joint_ratio(dry_error, dry_n),
               dry_false_positive_rate=joint_ratio(fp, dry_n), dry_gt_001=joint_ratio(dry_wet, dry_n))
    for index, (label, _, _) in enumerate(JOINT_BINS):
        row[f'region_recall_{label}'] = joint_ratio(stats[8 + index], joint_region_counts[lead][index])
    row['small_region_recall_1-20'] = joint_ratio(stats[8] + stats[9], sum(joint_region_counts[lead][:2]))
    joint_rows.append(row)
joint_calibration_results = pd.DataFrame(joint_rows)
joint_calibration_candidates = {}
for lead in (5, 10):
    table = joint_calibration_results.query('lead_min == @lead')
    ranked = table.sort_values(['CSI', 'recall', 'min_size', 'threshold'], ascending=[False, False, True, True])
    best = ranked.iloc[0]
    # Show an alternative within 0.01 absolute CSI of the best; this is a
    # review tradeoff, not an automatically selected operational setting.
    near_best = table[table.CSI >= best.CSI - 0.01]
    small_friendly = near_best.sort_values(
        ['small_region_recall_1-20', 'CSI'], ascending=False
    ).iloc[0]
    unfiltered = ranked[ranked.min_size == 0].iloc[0]
    joint_calibration_candidates[lead] = {'best_CSI': best.to_dict(), 'small_shower_tradeoff': small_friendly.to_dict()}
    print(f'\n+{lead} MINUTES — {joint_samples} validation samples')
    print('True region counts:', dict(zip([b[0] for b in JOINT_BINS], joint_region_counts[lead].tolist())))
    print('Top five by CSI:')
    print(ranked.head(5).to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    print('Review candidates:')
    print(pd.DataFrame([best, small_friendly, unfiltered], index=[
        'Best CSI', 'Small showers within 0.01 CSI', 'Best unfiltered'
    ]).to_string(float_format=lambda v: f'{v:.4f}'))
    if best.threshold in (min(JOINT_THRESHOLDS), max(JOINT_THRESHOLDS)) or best.min_size == max(JOINT_MIN_SIZES):
        print('Selected candidate touches a search boundary; review before freezing settings.')
print('\nMask metrics are pixel-pooled. Region recall means ANY overlap, not complete coverage.')
print('Small regions may include radar noise; region size is not rain intensity.')
print('MAE is normalized radar intensity. Candidates are validation-tuned, not test results.')
print('No settings were deployed or frozen. Review the tradeoff separately for each horizon.')


In [ ]:
# Calibration/strategy diagnostics below evaluate +5 minutes only.
# Use plot_validation_examples() above for paired +5/+10 forecast displays.
# Test dilation-based support-mask expansion on the existing checkpoint (no retraining).
from scipy import ndimage

STRONG_PROBABILITY_THRESHOLD = 0.40
STRONG_MIN_COMPONENT_SIZE = 25

DILATION_SUPPORT_SETTINGS = [
    {"name": "1px dilation, support>=0.10", "iterations": 1, "support_threshold": 0.10},
    {"name": "2px dilation, support>=0.10", "iterations": 2, "support_threshold": 0.10},
    {"name": "2px dilation, support>=0.15", "iterations": 2, "support_threshold": 0.15},
    {"name": "2px dilation, support>=0.20", "iterations": 2, "support_threshold": 0.20},
]

model2.eval()
dilation_probabilities = []
dilation_targets = []
dilation_raw_predictions = []

with torch.no_grad():
    for dilation_inputs, dilation_labels in val_loader:
        dilation_labels = dilation_labels[:, 0]  # Evaluate the primary +5-minute forecast.
        dilation_inputs = dilation_inputs.to(device)
        dilation_labels = dilation_labels.to(device)
        raw_outputs, _, _, dilation_logits = model2(dilation_inputs, return_logits=True)
        dilation_probabilities.append(torch.sigmoid(dilation_logits).squeeze(1).cpu())
        dilation_targets.append(dilation_labels.cpu())
        dilation_raw_predictions.append(raw_outputs.squeeze(1).clamp_min(0).cpu())

dilation_probabilities = torch.cat(dilation_probabilities)
dilation_targets = torch.cat(dilation_targets)
dilation_raw_predictions = torch.cat(dilation_raw_predictions)
dilation_rain_target = dilation_targets > 0.01


def remove_small_components(mask, minimum_size):
    structure = np.ones((3, 3), dtype=np.uint8)
    cleaned = []
    for sample_mask in mask.numpy():
        labels, _ = ndimage.label(sample_mask, structure=structure)
        sizes = np.bincount(labels.ravel())
        keep = sizes >= minimum_size
        keep[0] = False
        cleaned.append(torch.from_numpy(keep[labels]))
    return torch.stack(cleaned)


def build_support_mask(probabilities, iterations, support_threshold):
    strong_mask = remove_small_components(
        probabilities >= STRONG_PROBABILITY_THRESHOLD, STRONG_MIN_COMPONENT_SIZE
    )
    structure = np.ones((3, 3), dtype=np.uint8)
    expanded = torch.stack([
        torch.from_numpy(
            ndimage.binary_dilation(sample_mask, structure=structure, iterations=iterations)
        )
        for sample_mask in strong_mask.numpy()
    ])
    return expanded & (probabilities >= support_threshold)


def calculate_gated_metrics(mask, probabilities, raw_predictions, target_rain, targets):
    true_positive = (mask & target_rain).sum().item()
    false_positive = (mask & ~target_rain).sum().item()
    false_negative = (~mask & target_rain).sum().item()
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    csi = true_positive / max(true_positive + false_positive + false_negative, 1)
    f1 = 2.0 * precision * recall / max(precision + recall, 1e-12)
    gated = raw_predictions * mask.float()
    return {
        "precision": precision,
        "recall": recall,
        "csi": csi,
        "f1": f1,
        "rainy_mae": (gated[target_rain] - targets[target_rain]).abs().mean().item(),
        "dry_mae": (gated[~target_rain] - targets[~target_rain]).abs().mean().item(),
        "dry_gt_001": (gated[~target_rain] > 0.01).float().mean().item(),
    }


rain_pixel_counts = dilation_rain_target.sum(dim=(1, 2))
sorted_indices = torch.argsort(rain_pixel_counts, descending=True)
heavy_indices = sorted_indices[:5].tolist()
rainy_sorted = [idx for idx in sorted_indices.tolist() if rain_pixel_counts[idx] > 0]
light_indices = rainy_sorted[-5:]
example_indices = heavy_indices + light_indices
example_labels = ["Heavy"] * 5 + ["Light"] * 5

print("METRICS SUMMARY (full validation set)")
print("setting                          precision  recall  CSI     F1      rainy_MAE  dry_MAE  dry>.01")
for setting in DILATION_SUPPORT_SETTINGS:
    mask = build_support_mask(dilation_probabilities, setting["iterations"], setting["support_threshold"])
    metrics = calculate_gated_metrics(
        mask, dilation_probabilities, dilation_raw_predictions, dilation_rain_target, dilation_targets
    )
    setting["mask"] = mask
    setting["metrics"] = metrics
    print(
        f"{setting['name']:<32} {metrics['precision']:.4f}     {metrics['recall']:.4f}  "
        f"{metrics['csi']:.4f}  {metrics['f1']:.4f}  {metrics['rainy_mae']:.4f}     "
        f"{metrics['dry_mae']:.4f}   {metrics['dry_gt_001']:.4f}"
    )

# --- SINGLE COMBINED MONTAGE (all settings + all examples in one figure) ---
column_titles = ["Target", "Raw prediction", "Rain probability"] + [
    f"Gated: {s['name']}\nCSI={s['metrics']['csi']:.3f}" for s in DILATION_SUPPORT_SETTINGS
]
num_columns = len(column_titles)
num_rows = len(example_indices)

intensity_vmax = max(
    float(dilation_targets[example_indices].max()),
    float(dilation_raw_predictions[example_indices].max()),
    0.5,
)

combined_figure, combined_axes = plt.subplots(
    num_rows, num_columns, figsize=(4 * num_columns, 3.2 * num_rows), squeeze=False
)

for row_index, (sample_index, sample_label) in enumerate(zip(example_indices, example_labels)):
    display_number = row_index + 1 if row_index < 5 else row_index - 4
    row_images = [
        dilation_targets[sample_index].numpy(),
        dilation_raw_predictions[sample_index].numpy(),
        dilation_probabilities[sample_index].numpy(),
    ] + [
        (dilation_raw_predictions[sample_index] * s["mask"][sample_index].float()).numpy()
        for s in DILATION_SUPPORT_SETTINGS
    ]

    for column_index, panel_image in enumerate(row_images):
        panel_axis = combined_axes[row_index, column_index]
        if column_index == 2:
            panel_axis.imshow(panel_image, cmap="magma", vmin=0.0, vmax=1.0)
        else:
            panel_axis.imshow(panel_image, cmap="viridis", vmin=0.0, vmax=intensity_vmax)
        panel_axis.set_xticks([])
        panel_axis.set_yticks([])
        if row_index == 0:
            panel_axis.set_title(column_titles[column_index], fontsize=10)
        if column_index == 0:
            panel_axis.set_ylabel(f"{sample_label} #{display_number}", fontsize=10)

combined_figure.colorbar(
    plt.cm.ScalarMappable(norm=plt.Normalize(0.0, intensity_vmax), cmap="viridis"),
    ax=combined_axes[:, [0, 1] + list(range(3, num_columns))],
    fraction=0.02,
    pad=0.01,
    label="Rain intensity",
)
combined_figure.colorbar(
    plt.cm.ScalarMappable(norm=plt.Normalize(0.0, 1.0), cmap="magma"),
    ax=combined_axes[:, 2],
    fraction=0.08,
    pad=0.03,
    label="Rain probability",
)

combined_figure.suptitle(
    "Dilation-based support-mask sweep | 5 heavy + 5 light validation examples", fontsize=14
)
plt.show()

In [ ]:
# Calibration/strategy diagnostics below evaluate +5 minutes only.
# Use plot_validation_examples() above for paired +5/+10 forecast displays.

# =====================================================================================
# COMPREHENSIVE VALIDATION-ONLY MASK-CONFIGURATION EVALUATION
# Model weights, raw_prediction and preprocessing are NEVER modified. Validation
# (val_loader) only. Target is used only for metrics, never for mask construction.
# =====================================================================================
from scipy import ndimage
from scipy.ndimage import binary_dilation, uniform_filter
import numpy as np
import pandas as pd
import torch

STRUCTURE_8 = np.ones((3, 3), dtype=np.uint8)

# ---------------------------------------------------------------------------------
# 1. Collect validation predictions once: raw intensity, rain probability, target,
#    and the last input radar frame (for persistence comparison).
# ---------------------------------------------------------------------------------
model2.eval()
mcfg_targets, mcfg_raw, mcfg_prob, mcfg_last_radar = [], [], [], []

with torch.no_grad():
    for mcfg_inputs, mcfg_labels in val_loader:
        mcfg_labels = mcfg_labels[:, 0]  # Evaluate the primary +5-minute forecast.
        mcfg_inputs = mcfg_inputs.to(device)
        mcfg_labels = mcfg_labels.to(device)
        raw_out, _, _, rain_logits = model2(mcfg_inputs, return_logits=True)
        mcfg_raw.append(raw_out.squeeze(1).clamp_min(0).cpu())
        mcfg_prob.append(torch.sigmoid(rain_logits).squeeze(1).cpu())
        mcfg_targets.append(mcfg_labels.cpu())
        mcfg_last_radar.append(mcfg_inputs[:, -1, 0].cpu())  # last timestep, radar channel

mcfg_targets = torch.cat(mcfg_targets).numpy()
mcfg_raw = torch.cat(mcfg_raw).numpy()
mcfg_prob = torch.cat(mcfg_prob).numpy()
mcfg_last_radar = torch.cat(mcfg_last_radar).numpy()

NUM_SAMPLES = mcfg_targets.shape[0]
target_rain = mcfg_targets > 0.01
assert mcfg_raw.shape == mcfg_targets.shape == mcfg_prob.shape == mcfg_last_radar.shape
print(f"Collected {NUM_SAMPLES} validation samples ({mcfg_targets[0].size} pixels each) for mask evaluation.")

# ---------------------------------------------------------------------------------
# 2. Rain-regime classification per frame
# ---------------------------------------------------------------------------------
def classify_regime(fraction):
    if fraction <= 0.0:
        return "completely_dry"
    if fraction <= 0.005:
        return "very_light"
    if fraction <= 0.02:
        return "light"
    if fraction <= 0.10:
        return "moderate"
    return "heavy"

REGIME_ORDER = ["completely_dry", "very_light", "light", "moderate", "heavy"]
frame_rain_fraction = target_rain.reshape(NUM_SAMPLES, -1).mean(axis=1)
frame_regime = np.array([classify_regime(f) for f in frame_rain_fraction])

# ---------------------------------------------------------------------------------
# 3. Mask configurations (validation-only tuning; model/raw_prediction untouched)
# ---------------------------------------------------------------------------------
MASK_CONFIGS = [
    {"config": "A_baseline", "strong": 0.40, "min_size": 25, "dilation": 0, "support": None},
    {"config": "B", "strong": 0.40, "min_size": 25, "dilation": 1, "support": 0.10},
    {"config": "C", "strong": 0.40, "min_size": 25, "dilation": 1, "support": 0.15},
    {"config": "D", "strong": 0.40, "min_size": 25, "dilation": 1, "support": 0.20},
    {"config": "E", "strong": 0.40, "min_size": 25, "dilation": 2, "support": 0.10},
    {"config": "F", "strong": 0.40, "min_size": 25, "dilation": 2, "support": 0.15},
    {"config": "G", "strong": 0.40, "min_size": 25, "dilation": 2, "support": 0.20},
]


def clean_small_components(binary_stack, min_size):
    cleaned = np.zeros_like(binary_stack)
    for i in range(binary_stack.shape[0]):
        labels, _ = ndimage.label(binary_stack[i], structure=STRUCTURE_8)
        sizes = np.bincount(labels.ravel())
        keep = sizes >= min_size
        keep[0] = False
        cleaned[i] = keep[labels]
    return cleaned


def build_final_mask(rain_prob, cfg):
    strong = rain_prob >= cfg["strong"]
    cleaned_strong = clean_small_components(strong, cfg["min_size"])
    if cfg["dilation"] and cfg["dilation"] > 0:
        expanded = np.stack([
            binary_dilation(cleaned_strong[i], structure=STRUCTURE_8, iterations=cfg["dilation"])
            for i in range(cleaned_strong.shape[0])
        ])
    else:
        expanded = cleaned_strong
    if cfg["support"] is not None:
        return (expanded & (rain_prob >= cfg["support"])).astype(bool)
    return cleaned_strong.astype(bool)


def fractions_skill_score(pred_mask, targ_mask, window):
    box = 2 * window + 1
    numerator = 0.0
    denom = 0.0
    for i in range(pred_mask.shape[0]):
        pf = uniform_filter(pred_mask[i].astype(np.float64), size=box, mode="constant")
        tf = uniform_filter(targ_mask[i].astype(np.float64), size=box, mode="constant")
        numerator += np.sum((pf - tf) ** 2)
        denom += np.sum(pf ** 2) + np.sum(tf ** 2)
    return 1.0 if denom <= 0 else 1.0 - numerator / denom

# ---------------------------------------------------------------------------------
# 4. Precompute persistence baseline once (shared across all configs): for every true
#    rain component >=100 px, find the nearest last-radar-frame component centroid.
# ---------------------------------------------------------------------------------
persistence_lookup = {}  # (sample_index, target_component_id) -> persistence_centroid_error
target_labels_cache = []
target_counts_cache = []

for i in range(NUM_SAMPLES):
    t_labels, t_count = ndimage.label(target_rain[i], structure=STRUCTURE_8)
    target_labels_cache.append(t_labels)
    target_counts_cache.append(t_count)

    last_binary = mcfg_last_radar[i] > 0.01
    last_labels, last_count = ndimage.label(last_binary, structure=STRUCTURE_8)
    last_centroids = []
    for last_id in range(1, last_count + 1):
        lys, lxs = np.nonzero(last_labels == last_id)
        last_centroids.append((lys.mean(), lxs.mean()))

    for comp_id in range(1, t_count + 1):
        comp_mask = t_labels == comp_id
        if int(comp_mask.sum()) < 100:
            continue
        tys, txs = np.nonzero(comp_mask)
        t_cy, t_cx = tys.mean(), txs.mean()
        if last_centroids:
            dists = [float(np.hypot(cy - t_cy, cx - t_cx)) for cy, cx in last_centroids]
            persistence_lookup[(i, comp_id)] = min(dists)
        else:
            persistence_lookup[(i, comp_id)] = np.nan

# ---------------------------------------------------------------------------------
# 5. Per-configuration evaluation loop
# ---------------------------------------------------------------------------------
config_rows = []
component_rows = []
large_component_rows = []
regime_rows = []
config_masks = {}  # cache final_mask per config for the montage step later

INTENSITY_BINS = [(0.01, 0.05), (0.05, 0.10), (0.10, 0.20), (0.20, 0.40), (0.40, None)]
GE_THRESHOLDS = {"ge_100": 100, "ge_250": 250, "ge_500": 500}
TRUE_COMPONENT_BINS = [
    ("1-2", 1, 2), ("3-5", 3, 5), ("6-20", 6, 20), ("21-100", 21, 100),
    ("101-250", 101, 250), ("251-500", 251, 500), (">500", 501, None),
]
STANDALONE_SIZE_BINS = [(1, 2), (3, 5), (6, 10), (11, 24), (25, 49), (50, 99), (100, None)]

for cfg in MASK_CONFIGS:
    name = cfg["config"]
    final_mask = build_final_mask(mcfg_prob, cfg)
    config_masks[name] = final_mask
    final_pred = mcfg_raw * final_mask
    predicted_rain = final_pred > 0.01

    dry_mask_bool = ~target_rain
    dry_values = final_pred[dry_mask_bool]
    dry_targets = mcfg_targets[dry_mask_bool]
    dry_mae = float(np.abs(dry_values - dry_targets).mean())
    dry_mean_pred = float(dry_values.mean())
    dry_median_pred = float(np.median(dry_values))
    dry_p95_pred = float(np.percentile(dry_values, 95))
    dry_p99_pred = float(np.percentile(dry_values, 99))
    dry_max_pred = float(dry_values.max())
    dry_gt_001 = float((dry_values > 0.01).mean())
    dry_gt_002 = float((dry_values > 0.02).mean())
    dry_gt_005 = float((dry_values > 0.05).mean())

    completely_dry_idx = np.where(frame_regime == "completely_dry")[0]
    if len(completely_dry_idx) > 0:
        false_rain_fracs = predicted_rain[completely_dry_idx].reshape(len(completely_dry_idx), -1).mean(axis=1)
        cdry_mean_false = float(false_rain_fracs.mean())
        cdry_median_false = float(np.median(false_rain_fracs))
        cdry_p95_false = float(np.percentile(false_rain_fracs, 95))
        cdry_max_false = float(false_rain_fracs.max())
        cdry_clean_fraction = float((false_rain_fracs == 0).mean())
        cdry_component_counts = [
            ndimage.label(predicted_rain[i], structure=STRUCTURE_8)[1] for i in completely_dry_idx
        ]
        cdry_mean_components = float(np.mean(cdry_component_counts))
    else:
        cdry_mean_false = cdry_median_false = cdry_p95_false = cdry_max_false = np.nan
        cdry_clean_fraction = np.nan
        cdry_mean_components = np.nan

    tp = int((final_mask & target_rain).sum())
    fp = int((final_mask & dry_mask_bool).sum())
    fn = int((~final_mask & target_rain).sum())
    tn = int((~final_mask & dry_mask_bool).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    csi = tp / max(tp + fp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    specificity = tn / max(tn + fp, 1)
    false_positive_rate = fp / max(fp + tn, 1)
    false_negative_rate = fn / max(fn + tp, 1)

    macro_p, macro_r, macro_c, macro_f = [], [], [], []
    for i in range(NUM_SAMPLES):
        if frame_regime[i] == "completely_dry":
            continue
        tp_i = int((final_mask[i] & target_rain[i]).sum())
        fp_i = int((final_mask[i] & ~target_rain[i]).sum())
        fn_i = int((~final_mask[i] & target_rain[i]).sum())
        p_i = tp_i / max(tp_i + fp_i, 1)
        r_i = tp_i / max(tp_i + fn_i, 1)
        c_i = tp_i / max(tp_i + fp_i + fn_i, 1)
        f_i = 2 * p_i * r_i / max(p_i + r_i, 1e-12)
        macro_p.append(p_i); macro_r.append(r_i); macro_c.append(c_i); macro_f.append(f_i)
    macro_precision = float(np.mean(macro_p)) if macro_p else np.nan
    macro_recall = float(np.mean(macro_r)) if macro_r else np.nan
    macro_csi = float(np.mean(macro_c)) if macro_c else np.nan
    macro_f1 = float(np.mean(macro_f)) if macro_f else np.nan

    standalone_total = 0
    standalone_pixels_total = 0
    spillover_pixels_total = 0
    standalone_sizes = []
    frames_with_standalone = 0
    detection_counts = {b[0]: [0, 0] for b in TRUE_COMPONENT_BINS}
    ge_counts = {k: [0, 0] for k in GE_THRESHOLDS}

    for i in range(NUM_SAMPLES):
        pr_labels, pr_count = ndimage.label(predicted_rain[i], structure=STRUCTURE_8)
        frame_has_standalone = False
        for comp_id in range(1, pr_count + 1):
            comp_mask = pr_labels == comp_id
            comp_size = int(comp_mask.sum())
            tp_pixels = int((comp_mask & target_rain[i]).sum())
            fp_pixels = comp_size - tp_pixels
            comp_precision = tp_pixels / max(comp_size, 1)
            standalone = tp_pixels == 0
            ys, xs = np.nonzero(comp_mask)
            centroid_y, centroid_x = float(ys.mean()), float(xs.mean())
            h, w = predicted_rain.shape[1], predicted_rain.shape[2]
            edge_distance = float(min(centroid_y, h - 1 - centroid_y, centroid_x, w - 1 - centroid_x))
            component_rows.append({
                "config": name, "sample_index": i, "component_id": comp_id, "component_size": comp_size,
                "tp_pixels": tp_pixels, "fp_pixels": fp_pixels, "component_precision": comp_precision,
                "standalone": bool(standalone), "centroid_y": centroid_y, "centroid_x": centroid_x,
                "edge_distance": edge_distance,
            })
            if standalone:
                standalone_total += 1
                standalone_pixels_total += fp_pixels
                standalone_sizes.append(comp_size)
                frame_has_standalone = True
            else:
                spillover_pixels_total += fp_pixels
        if frame_has_standalone:
            frames_with_standalone += 1

        fm_labels, fm_count = ndimage.label(final_mask[i], structure=STRUCTURE_8)
        t_labels = target_labels_cache[i]
        t_count = target_counts_cache[i]
        for comp_id in range(1, t_count + 1):
            comp_mask = t_labels == comp_id
            comp_size = int(comp_mask.sum())
            detected = bool(np.any(final_mask[i] & comp_mask))
            for bin_name, lo, hi in TRUE_COMPONENT_BINS:
                if comp_size >= lo and (hi is None or comp_size <= hi):
                    detection_counts[bin_name][1] += 1
                    detection_counts[bin_name][0] += int(detected)
                    break
            for ge_name, thresh in GE_THRESHOLDS.items():
                if comp_size >= thresh:
                    ge_counts[ge_name][1] += 1
                    ge_counts[ge_name][0] += int(detected)

            if comp_size >= 100:
                overlap_labels = fm_labels[comp_mask]
                overlap_labels = overlap_labels[overlap_labels > 0]
                if overlap_labels.size == 0:
                    matched_id, predicted_size, intersection = None, 0, 0
                else:
                    vals, counts = np.unique(overlap_labels, return_counts=True)
                    matched_id = int(vals[np.argmax(counts)])
                    intersection = int(counts.max())
                    predicted_size = int((fm_labels == matched_id).sum())
                union = comp_size + predicted_size - intersection
                iou = intersection / max(union, 1)
                dice = 2 * intersection / max(comp_size + predicted_size, 1)
                area_ratio = predicted_size / max(comp_size, 1)
                abs_area_error = abs(predicted_size - comp_size)
                tys, txs = np.nonzero(comp_mask)
                t_cy, t_cx = float(tys.mean()), float(txs.mean())
                if matched_id is not None:
                    pys, pxs = np.nonzero(fm_labels == matched_id)
                    p_cy, p_cx = float(pys.mean()), float(pxs.mean())
                    forecast_centroid_error = float(np.hypot(p_cy - t_cy, p_cx - t_cx))
                else:
                    forecast_centroid_error = np.nan
                persistence_error = persistence_lookup.get((i, comp_id), np.nan)
                large_component_rows.append({
                    "config": name, "sample_index": i, "target_component_id": comp_id,
                    "target_size": comp_size, "detected": int(matched_id is not None),
                    "matched_pred_component_id": matched_id, "predicted_size": predicted_size,
                    "intersection_pixels": intersection, "IoU": iou, "Dice": dice,
                    "area_ratio": area_ratio, "absolute_area_error": abs_area_error,
                    "centroid_distance": forecast_centroid_error,
                    "persistence_centroid_error": persistence_error,
                    "forecast_centroid_error": forecast_centroid_error,
                })

    total_fp_pixels = standalone_pixels_total + spillover_pixels_total
    standalone_fraction_of_fp = standalone_pixels_total / max(total_fp_pixels, 1)
    spillover_fraction_of_fp = spillover_pixels_total / max(total_fp_pixels, 1)
    largest_standalone = max(standalone_sizes) if standalone_sizes else 0
    median_standalone = float(np.median(standalone_sizes)) if standalone_sizes else 0.0
    p95_standalone = float(np.percentile(standalone_sizes, 95)) if standalone_sizes else 0.0
    standalone_size_hist = {
        f"standalone_{lo}-{hi if hi else 'plus'}": sum(
            1 for s in standalone_sizes if s >= lo and (hi is None or s <= hi)
        )
        for lo, hi in STANDALONE_SIZE_BINS
    }

    detection_rates = {b: (detection_counts[b][0] / max(detection_counts[b][1], 1)) for b, _, _ in TRUE_COMPONENT_BINS}
    detection_rate_ge_100 = ge_counts["ge_100"][0] / max(ge_counts["ge_100"][1], 1)
    detection_rate_ge_250 = ge_counts["ge_250"][0] / max(ge_counts["ge_250"][1], 1)
    detection_rate_ge_500 = ge_counts["ge_500"][0] / max(ge_counts["ge_500"][1], 1)

    cfg_large_rows = [r for r in large_component_rows if r["config"] == name]
    def agg_stats(values):
        if not values:
            return (np.nan, np.nan, np.nan, np.nan)
        arr = np.array(values, dtype=float)
        arr = arr[~np.isnan(arr)]
        if arr.size == 0:
            return (np.nan, np.nan, np.nan, np.nan)
        return (float(arr.mean()), float(np.median(arr)), float(np.percentile(arr, 25)), float(np.percentile(arr, 75)))

    large_iou_mean, large_iou_median, large_iou_p25, large_iou_p75 = agg_stats([r["IoU"] for r in cfg_large_rows])
    large_dice_mean, *_ = agg_stats([r["Dice"] for r in cfg_large_rows])
    large_area_ratio_mean, *_ = agg_stats([r["area_ratio"] for r in cfg_large_rows])
    large_area_error_mean, *_ = agg_stats([r["absolute_area_error"] for r in cfg_large_rows])
    large_centroid_mean, large_centroid_median, *_ = agg_stats([r["forecast_centroid_error"] for r in cfg_large_rows])

    motion_deltas, forecast_errors, persistence_errors = [], [], []
    for r in cfg_large_rows:
        fe, pe = r["forecast_centroid_error"], r["persistence_centroid_error"]
        if fe is not None and pe is not None and not (isinstance(fe, float) and np.isnan(fe)) and not (isinstance(pe, float) and np.isnan(pe)):
            motion_deltas.append(pe - fe)
            forecast_errors.append(fe)
            persistence_errors.append(pe)
    if motion_deltas:
        median_persistence_error = float(np.median(persistence_errors))
        median_forecast_error = float(np.median(forecast_errors))
        mean_persistence_error = float(np.mean(persistence_errors))
        mean_forecast_error = float(np.mean(forecast_errors))
        fraction_beats_persistence = float(np.mean([d > 0 for d in motion_deltas]))
        mean_motion_improvement = float(np.mean(motion_deltas))
    else:
        median_persistence_error = median_forecast_error = np.nan
        mean_persistence_error = mean_forecast_error = np.nan
        fraction_beats_persistence = np.nan
        mean_motion_improvement = np.nan

    fss_1 = fractions_skill_score(final_mask, target_rain, window=1)
    fss_3 = fractions_skill_score(final_mask, target_rain, window=3)
    fss_5 = fractions_skill_score(final_mask, target_rain, window=5)
    fss_10 = fractions_skill_score(final_mask, target_rain, window=10)

    overall_mae = float(np.abs(final_pred - mcfg_targets).mean())
    rainy_mae = float(np.abs(final_pred[target_rain] - mcfg_targets[target_rain]).mean()) if target_rain.any() else np.nan
    rainy_mae_by_bin = {}
    for lo, hi in INTENSITY_BINS:
        if hi is None:
            sel = (mcfg_targets > lo) & target_rain
        else:
            sel = (mcfg_targets > lo) & (mcfg_targets <= hi) & target_rain
        key = f"rainy_mae_{lo}-{hi if hi is not None else 'plus'}"
        rainy_mae_by_bin[key] = float(np.abs(final_pred[sel] - mcfg_targets[sel]).mean()) if sel.sum() > 0 else np.nan
    mean_target_rain_intensity = float(mcfg_targets[target_rain].mean()) if target_rain.any() else np.nan
    mean_predicted_rain_intensity = float(final_pred[target_rain].mean()) if target_rain.any() else np.nan
    rain_intensity_ratio = mean_predicted_rain_intensity / max(mean_target_rain_intensity, 1e-12)

    raw_binary = mcfg_raw > 0.01
    raw_tp = int((raw_binary & target_rain).sum())
    raw_fp = int((raw_binary & ~target_rain).sum())
    raw_fn = int((~raw_binary & target_rain).sum())
    raw_precision = raw_tp / max(raw_tp + raw_fp, 1)
    raw_recall = raw_tp / max(raw_tp + raw_fn, 1)
    raw_csi = raw_tp / max(raw_tp + raw_fp + raw_fn, 1)
    raw_f1 = 2 * raw_precision * raw_recall / max(raw_precision + raw_recall, 1e-12)
    raw_rainy_mae = float(np.abs(mcfg_raw[target_rain] - mcfg_targets[target_rain]).mean()) if target_rain.any() else np.nan
    raw_dry_mae = float(np.abs(mcfg_raw[dry_mask_bool] - mcfg_targets[dry_mask_bool]).mean())
    raw_dry_gt_001 = float((mcfg_raw[dry_mask_bool] > 0.01).mean())
    raw_dry_gt_005 = float((mcfg_raw[dry_mask_bool] > 0.05).mean())

    for regime in REGIME_ORDER:
        idx = np.where(frame_regime == regime)[0]
        if len(idx) == 0:
            continue
        r_target_rain = target_rain[idx]
        r_final_mask = final_mask[idx]
        r_final_pred = final_pred[idx]
        r_targets = mcfg_targets[idx]
        tp_r = int((r_final_mask & r_target_rain).sum())
        fp_r = int((r_final_mask & ~r_target_rain).sum())
        fn_r = int((~r_final_mask & r_target_rain).sum())
        p_r = tp_r / max(tp_r + fp_r, 1)
        rc_r = tp_r / max(tp_r + fn_r, 1)
        c_r = tp_r / max(tp_r + fp_r + fn_r, 1)
        f_r = 2 * p_r * rc_r / max(p_r + rc_r, 1e-12)
        rainy_mae_r = float(np.abs(r_final_pred[r_target_rain] - r_targets[r_target_rain]).mean()) if r_target_rain.any() else np.nan
        dry_mae_r = float(np.abs(r_final_pred[~r_target_rain] - r_targets[~r_target_rain]).mean())
        dry_gt_001_r = float((r_final_pred[~r_target_rain] > 0.01).mean())
        standalone_count_regime = 0
        for i in idx:
            labels_i, count_i = ndimage.label(predicted_rain[i], structure=STRUCTURE_8)
            for comp_id in range(1, count_i + 1):
                if (labels_i == comp_id).astype(bool)[target_rain[i]].sum() == 0:
                    standalone_count_regime += 1
        regime_rows.append({
            "config": name, "regime": regime, "n_frames": len(idx),
            "precision": p_r, "recall": rc_r, "csi": c_r, "f1": f_r,
            "rainy_mae": rainy_mae_r, "dry_mae": dry_mae_r, "dry_gt_001": dry_gt_001_r,
            "standalone_false_components_per_frame": standalone_count_regime / len(idx),
        })

    row = {
        "config": name, "strong_threshold": cfg["strong"], "min_component_size": cfg["min_size"],
        "dilation_iterations": cfg["dilation"], "support_threshold": cfg["support"],
        "precision": precision, "recall": recall, "csi": csi, "f1": f1,
        "specificity": specificity, "false_positive_rate": false_positive_rate, "false_negative_rate": false_negative_rate,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "macro_precision": macro_precision, "macro_recall": macro_recall, "macro_csi": macro_csi, "macro_f1": macro_f1,
        "dry_mae": dry_mae, "dry_mean_prediction": dry_mean_pred, "dry_median_prediction": dry_median_pred,
        "dry_p95_prediction": dry_p95_pred, "dry_p99_prediction": dry_p99_pred, "dry_max_prediction": dry_max_pred,
        "dry_gt_001": dry_gt_001, "dry_gt_002": dry_gt_002, "dry_gt_005": dry_gt_005,
        "completely_dry_mean_false_rain_fraction": cdry_mean_false, "completely_dry_median_false_rain_fraction": cdry_median_false,
        "completely_dry_p95_false_rain_fraction": cdry_p95_false, "completely_dry_max_false_rain_fraction": cdry_max_false,
        "completely_dry_clean_frame_fraction": cdry_clean_fraction, "completely_dry_mean_predicted_components": cdry_mean_components,
        "standalone_false_components_total": standalone_total, "standalone_false_components_per_frame": standalone_total / NUM_SAMPLES,
        "standalone_false_pixels_total": standalone_pixels_total, "standalone_fraction_of_all_FP": standalone_fraction_of_fp,
        "spillover_fraction_of_all_FP": spillover_fraction_of_fp,
        "largest_standalone_component": largest_standalone, "median_standalone_component_size": median_standalone,
        "p95_standalone_component_size": p95_standalone, "frames_with_standalone_component": frames_with_standalone,
        **standalone_size_hist,
        **{f"detection_rate_{b}": detection_rates[b] for b in detection_rates},
        "detection_rate_ge_100": detection_rate_ge_100, "detection_rate_ge_250": detection_rate_ge_250, "detection_rate_ge_500": detection_rate_ge_500,
        "large_component_iou_mean": large_iou_mean, "large_component_iou_median": large_iou_median,
        "large_component_iou_p25": large_iou_p25, "large_component_iou_p75": large_iou_p75,
        "large_component_dice_mean": large_dice_mean, "large_component_area_ratio_mean": large_area_ratio_mean,
        "large_component_abs_area_error_mean": large_area_error_mean,
        "large_component_centroid_error_mean": large_centroid_mean, "large_component_centroid_error_median": large_centroid_median,
        "median_persistence_centroid_error": median_persistence_error, "median_forecast_centroid_error": median_forecast_error,
        "mean_persistence_centroid_error": mean_persistence_error, "mean_forecast_centroid_error": mean_forecast_error,
        "fraction_matched_beats_persistence": fraction_beats_persistence, "mean_motion_improvement": mean_motion_improvement,
        "fss_1": fss_1, "fss_3": fss_3, "fss_5": fss_5, "fss_10": fss_10,
        "overall_mae": overall_mae, "rainy_mae": rainy_mae, **rainy_mae_by_bin,
        "mean_target_rain_intensity": mean_target_rain_intensity, "mean_predicted_rain_intensity": mean_predicted_rain_intensity,
        "rain_intensity_ratio": rain_intensity_ratio,
        "raw_precision": raw_precision, "raw_recall": raw_recall, "raw_csi": raw_csi, "raw_f1": raw_f1,
        "raw_rainy_mae": raw_rainy_mae, "raw_dry_mae": raw_dry_mae, "raw_dry_gt_001": raw_dry_gt_001, "raw_dry_gt_005": raw_dry_gt_005,
    }
    config_rows.append(row)
    print(f"Finished evaluating configuration {name}")

config_df = pd.DataFrame(config_rows)
component_df = pd.DataFrame(component_rows)
large_component_df = pd.DataFrame(large_component_rows)
regime_df = pd.DataFrame(regime_rows)

config_df.to_csv("mask_configuration_validation_report.csv", index=False)
component_df.to_csv("mask_component_validation_report.csv", index=False)
large_component_df.to_csv("mask_large_rain_component_report.csv", index=False)
regime_df.to_csv("mask_regime_validation_report.csv", index=False)
print("Saved: mask_configuration_validation_report.csv, mask_component_validation_report.csv, "
      "mask_large_rain_component_report.csv, mask_regime_validation_report.csv")

# ---------------------------------------------------------------------------------
# 6. Terminal ranked summary table
# ---------------------------------------------------------------------------------
print("\nCOMPACT RANKED SUMMARY (sorted by CSI, descending)")
header = (
    f"{'Config':<11}{'Strong':>7}{'MinSz':>6}{'Dil':>4}{'Supp':>6}"
    f"{'Prec':>8}{'Rec':>8}{'CSI':>8}{'F1':>8}{'Dry>.01':>9}{'DryMAE':>9}"
    f"{'StandFP':>9}{'>=100':>7}{'>=250':>7}{'>=500':>7}{'LgIoU':>8}{'CentErr':>9}{'RainyMAE':>10}{'FSS5':>7}"
)
print(header)
for _, r in config_df.sort_values("csi", ascending=False).iterrows():
    support_str = f"{r['support_threshold']:.2f}" if pd.notna(r["support_threshold"]) else "None"
    print(
        f"{r['config']:<11}{r['strong_threshold']:>7.2f}{r['min_component_size']:>6.0f}{r['dilation_iterations']:>4.0f}{support_str:>6}"
        f"{r['precision']:>8.4f}{r['recall']:>8.4f}{r['csi']:>8.4f}{r['f1']:>8.4f}{r['dry_gt_001']:>9.4f}{r['dry_mae']:>9.4f}"
        f"{r['standalone_false_components_per_frame']:>9.3f}{r['detection_rate_ge_100']:>7.3f}{r['detection_rate_ge_250']:>7.3f}"
        f"{r['detection_rate_ge_500']:>7.3f}{r['large_component_iou_mean']:>8.4f}{r['large_component_centroid_error_mean']:>9.3f}"
        f"{r['rainy_mae']:>10.4f}{r['fss_5']:>7.4f}"
    )

# ---------------------------------------------------------------------------------
# 7. Selection logic: hard requirements first, then Pareto-style comparison.
#    Priority: background cleanliness > noise rejection > large-rain preservation
#    > spatial/motion quality > overall CSI/F1 > intensity accuracy > tiny components.
# ---------------------------------------------------------------------------------
baseline_row = config_df[config_df["config"] == "A_baseline"].iloc[0]

candidates = config_df[config_df["config"] != "A_baseline"].copy()
candidates = candidates[
    (candidates["dry_gt_001"] <= max(baseline_row["dry_gt_001"] * 1.5, 0.01))
    & (candidates["dry_mae"] <= baseline_row["dry_mae"] * 1.5 + 1e-6)
    & (candidates["large_component_iou_mean"] >= baseline_row["large_component_iou_mean"] * 0.95)
    & (candidates["rainy_mae"] <= baseline_row["rainy_mae"] * 1.2)
]

rejected = config_df[
    (config_df["config"] != "A_baseline") & (~config_df["config"].isin(candidates["config"]))
]

if candidates.empty:
    recommended_row = baseline_row
    recommended_name = "A_baseline"
else:
    candidates = candidates.sort_values(
        ["csi", "detection_rate_ge_100", "large_component_iou_mean"], ascending=False
    )
    recommended_row = candidates.iloc[0]
    recommended_name = recommended_row["config"]

highest_csi_row = config_df.sort_values("csi", ascending=False).iloc[0]

print("\n" + "=" * 70)
print("RECOMMENDED CONFIGURATION")
print("=" * 70)
print(f"strong threshold: {recommended_row['strong_threshold']:.2f}")
print(f"min component size: {int(recommended_row['min_component_size'])}")
print(f"dilation iterations: {int(recommended_row['dilation_iterations'])}")
support_display = f"{recommended_row['support_threshold']:.2f}" if pd.notna(recommended_row["support_threshold"]) else "None"
print(f"support threshold: {support_display}")

print("\nBackground cleanliness:")
print(
    f"  dry>0.01={recommended_row['dry_gt_001']:.4f} (baseline={baseline_row['dry_gt_001']:.4f}), "
    f"dry_mae={recommended_row['dry_mae']:.4f} (baseline={baseline_row['dry_mae']:.4f})"
)
print("Noise rejection:")
print(
    f"  standalone false components/frame={recommended_row['standalone_false_components_per_frame']:.3f} "
    f"(baseline={baseline_row['standalone_false_components_per_frame']:.3f})"
)
print("Large-rain preservation:")
print(
    f"  detection>=100px={recommended_row['detection_rate_ge_100']:.3f}, "
    f">=250px={recommended_row['detection_rate_ge_250']:.3f}, >=500px={recommended_row['detection_rate_ge_500']:.3f} "
    f"(baseline: {baseline_row['detection_rate_ge_100']:.3f}/{baseline_row['detection_rate_ge_250']:.3f}/{baseline_row['detection_rate_ge_500']:.3f})"
)
print("Spatial/motion quality:")
print(
    f"  large-component IoU={recommended_row['large_component_iou_mean']:.4f} "
    f"(baseline={baseline_row['large_component_iou_mean']:.4f}), "
    f"centroid error={recommended_row['large_component_centroid_error_mean']:.3f}px, "
    f"fraction beating persistence={recommended_row['fraction_matched_beats_persistence']:.3f}"
)
print("Intensity trade-off:")
print(
    f"  rainy_mae={recommended_row['rainy_mae']:.4f} (baseline={baseline_row['rainy_mae']:.4f}), "
    f"rain_intensity_ratio={recommended_row['rain_intensity_ratio']:.3f}"
)
print("Why this configuration was preferred:")
print(
    f"  It satisfies the hard background/noise/large-rain constraints (dry_gt_001, dry_mae, "
    f"large-component IoU and rainy MAE within tolerance of baseline) while maximizing CSI, "
    f">=100px detection and large-component IoU among the surviving candidates."
)
if highest_csi_row["config"] != recommended_name:
    print(
        f"\nNote: '{highest_csi_row['config']}' had the single highest CSI ({highest_csi_row['csi']:.4f}) "
        f"but was REJECTED because it violated a hard structural requirement "
        f"(dry_gt_001={highest_csi_row['dry_gt_001']:.4f}, dry_mae={highest_csi_row['dry_mae']:.4f}, "
        f"large-component IoU={highest_csi_row['large_component_iou_mean']:.4f}, "
        f"rainy_mae={highest_csi_row['rainy_mae']:.4f}) relative to the baseline tolerances."
    )
if not rejected.empty:
    print("\nOther rejected configurations (failed hard structural requirements):")
    for _, r in rejected.iterrows():
        print(f"  {r['config']}: CSI={r['csi']:.4f}, dry_gt_001={r['dry_gt_001']:.4f}, dry_mae={r['dry_mae']:.4f}")

# ---------------------------------------------------------------------------------
# 8. Montage: baseline vs recommended candidate, 5 examples per regime group.
# ---------------------------------------------------------------------------------
baseline_mask = config_masks["A_baseline"]
candidate_mask = config_masks[recommended_name]
baseline_pred = mcfg_raw * baseline_mask
candidate_pred = mcfg_raw * candidate_mask

montage_groups = [
    ("Dry/near-dry", np.where(np.isin(frame_regime, ["completely_dry", "very_light"]))[0]),
    ("Light", np.where(frame_regime == "light")[0]),
    ("Moderate", np.where(frame_regime == "moderate")[0]),
    ("Heavy", np.where(frame_regime == "heavy")[0]),
]

montage_indices, montage_labels = [], []
for group_name, idx_pool in montage_groups:
    chosen = idx_pool[:5]
    montage_indices.extend(chosen.tolist())
    montage_labels.extend([f"{group_name} #{n + 1}" for n in range(len(chosen))])

montage_columns = ["Target", "Rain probability", "Baseline mask", "Baseline prediction", "Candidate mask", "Candidate prediction"]
mfig, maxes = plt.subplots(
    len(montage_indices), len(montage_columns),
    figsize=(3.6 * len(montage_columns), 3.0 * len(montage_indices)), squeeze=False,
)
for row_i, (sample_i, row_label) in enumerate(zip(montage_indices, montage_labels)):
    vmax = max(float(mcfg_targets[sample_i].max()), float(baseline_pred[sample_i].max()), float(candidate_pred[sample_i].max()), 0.5)
    row_panels = [
        (mcfg_targets[sample_i], "viridis", 0.0, vmax),
        (mcfg_prob[sample_i], "magma", 0.0, 1.0),
        (baseline_mask[sample_i].astype(float), "gray", 0.0, 1.0),
        (baseline_pred[sample_i], "viridis", 0.0, vmax),
        (candidate_mask[sample_i].astype(float), "gray", 0.0, 1.0),
        (candidate_pred[sample_i], "viridis", 0.0, vmax),
    ]
    for col_i, (image, cmap, vmin, vm) in enumerate(row_panels):
        axis = maxes[row_i, col_i]
        axis.imshow(image, cmap=cmap, vmin=vmin, vmax=vm)
        axis.set_xticks([]); axis.set_yticks([])
        if row_i == 0:
            axis.set_title(montage_columns[col_i], fontsize=10)
        if col_i == 0:
            axis.set_ylabel(row_label, fontsize=9)
mfig.suptitle(f"Baseline (A) vs recommended candidate ({recommended_name})", fontsize=14)
mfig.tight_layout()
mfig.savefig("mask_baseline_vs_candidate_montage.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------------
# 9. Diagnostic trade-off plots
# ---------------------------------------------------------------------------------
tfig, taxes = plt.subplots(2, 2, figsize=(13, 11))
plot_specs = [
    (taxes[0, 0], "dry_gt_001", "csi", "dry_gt_001", "CSI"),
    (taxes[0, 1], "precision", "recall", "Precision", "Recall"),
    (taxes[1, 0], "dry_gt_001", "large_component_iou_mean", "dry_gt_001", "Large-component IoU"),
    (taxes[1, 1], "dry_gt_001", "rainy_mae", "dry_gt_001", "Rainy MAE"),
]
for axis, x_col, y_col, x_label, y_label in plot_specs:
    axis.scatter(config_df[x_col], config_df[y_col])
    for _, r in config_df.iterrows():
        axis.annotate(r["config"], (r[x_col], r[y_col]), fontsize=8, xytext=(3, 3), textcoords="offset points")
    axis.set_xlabel(x_label)
    axis.set_ylabel(y_label)
    axis.grid(True, alpha=0.3)
tfig.suptitle("Mask configuration trade-offs (validation set)", fontsize=14)
tfig.tight_layout()
tfig.savefig("mask_configuration_tradeoffs.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Calibration/strategy diagnostics below evaluate +5 minutes only.
# Use plot_validation_examples() above for paired +5/+10 forecast displays.

# =====================================================================================
# EXTENDED VALIDATION-ONLY MASK STRATEGY SWEEP (conservative post-processing variants)
# Reuses mcfg_prob / mcfg_raw / mcfg_targets / target_rain / frame_regime /
# persistence_lookup / target_labels_cache / config_masks / config_df computed in the
# previous mask-configuration evaluation cell. Model weights, raw_prediction and
# preprocessing are NEVER modified. Validation (val_loader) data only.
# =====================================================================================
from scipy import ndimage
from scipy.ndimage import binary_dilation, binary_closing
import numpy as np
import pandas as pd

assert "mcfg_prob" in globals(), "Run the previous mask-configuration evaluation cell first."

# ---------------------------------------------------------------------------------
# Mask-construction strategies
# ---------------------------------------------------------------------------------
def ext_fill_holes(mask_2d, max_hole_size):
    filled = mask_2d.copy()
    inverse_labels, inverse_count = ndimage.label(~mask_2d, structure=STRUCTURE_8)
    for label_id in range(1, inverse_count + 1):
        comp = inverse_labels == label_id
        touches_border = comp[0, :].any() or comp[-1, :].any() or comp[:, 0].any() or comp[:, -1].any()
        if touches_border:
            continue
        if int(comp.sum()) <= max_hole_size:
            filled[comp] = True
    return filled


def build_hard_threshold_mask(threshold, min_size=25):
    return clean_small_components(mcfg_prob >= threshold, min_size).astype(bool)


def build_fill_holes_mask(max_hole_size, threshold=0.40, min_size=25):
    cleaned = clean_small_components(mcfg_prob >= threshold, min_size)
    return np.stack([ext_fill_holes(cleaned[i], max_hole_size) for i in range(cleaned.shape[0])]).astype(bool)


def build_closing_mask(iterations, threshold=0.40, min_size=25):
    cleaned = clean_small_components(mcfg_prob >= threshold, min_size)
    return np.stack([
        binary_closing(cleaned[i], structure=STRUCTURE_8, iterations=iterations)
        for i in range(cleaned.shape[0])
    ]).astype(bool)


def build_large_component_dilation_mask(min_large_size, iterations=1, threshold=0.40, min_size=25,
                                         support_kind=None, support_prob=None, support_raw=None,
                                         hole_fill=None):
    cleaned = clean_small_components(mcfg_prob >= threshold, min_size)
    if hole_fill is not None:
        cleaned = np.stack([ext_fill_holes(cleaned[i], hole_fill) for i in range(cleaned.shape[0])])
    result = np.zeros_like(cleaned)
    for i in range(cleaned.shape[0]):
        labels, count = ndimage.label(cleaned[i], structure=STRUCTURE_8)
        frame_result = np.zeros(cleaned[i].shape, dtype=bool)
        for comp_id in range(1, count + 1):
            base = labels == comp_id
            if int(base.sum()) >= min_large_size:
                expanded = binary_dilation(base, structure=STRUCTURE_8, iterations=iterations)
                extra = expanded & ~base
                if support_kind == "prob":
                    extra = extra & (mcfg_prob[i] >= support_prob)
                elif support_kind == "raw":
                    extra = extra & (mcfg_raw[i] >= support_raw)
                elif support_kind == "combined":
                    extra = extra & (mcfg_prob[i] >= support_prob) & (mcfg_raw[i] >= support_raw)
                frame_result |= base | extra
            else:
                frame_result |= base
        result[i] = frame_result
    return result.astype(bool)

# ---------------------------------------------------------------------------------
# Build the configuration list (baseline excluded here; it is already in config_df)
# ---------------------------------------------------------------------------------
EXTENDED_CONFIGS = []

for cname, thr in [("H1", 0.30), ("H2", 0.35), ("H3", 0.375), ("H4", 0.40), ("H5", 0.425), ("H6", 0.45), ("H7", 0.50)]:
    EXTENDED_CONFIGS.append({
        "config": cname, "method": "hard_threshold", "threshold": thr, "min_size": 25,
        "hole_fill": None, "large_cutoff": None, "support_prob": None, "support_raw": None,
        "builder": (lambda thr=thr: build_hard_threshold_mask(thr, 25)),
    })

for cname, size in [("FILL5", 5), ("FILL10", 10), ("FILL20", 20), ("FILL30", 30), ("FILL50", 50)]:
    EXTENDED_CONFIGS.append({
        "config": cname, "method": "hole_fill", "threshold": 0.40, "min_size": 25,
        "hole_fill": size, "large_cutoff": None, "support_prob": None, "support_raw": None,
        "builder": (lambda size=size: build_fill_holes_mask(size)),
    })

for cname, iters in [("CLOSE3", 1), ("CLOSE3_2", 2)]:
    EXTENDED_CONFIGS.append({
        "config": cname, "method": "morph_closing", "threshold": 0.40, "min_size": 25,
        "hole_fill": None, "large_cutoff": None, "support_prob": None, "support_raw": None,
        "builder": (lambda iters=iters: build_closing_mask(iters)),
    })

for cutoff in [100, 250, 500, 1000]:
    EXTENDED_CONFIGS.append({
        "config": f"LD{cutoff}", "method": "large_component_dilation", "threshold": 0.40, "min_size": 25,
        "hole_fill": None, "large_cutoff": cutoff, "support_prob": None, "support_raw": None,
        "builder": (lambda cutoff=cutoff: build_large_component_dilation_mask(cutoff)),
    })

for cutoff in [100, 250, 500, 1000]:
    for support in [0.15, 0.20, 0.25, 0.30]:
        EXTENDED_CONFIGS.append({
            "config": f"LD{cutoff}_P{support:.2f}", "method": "large_component_prob_support",
            "threshold": 0.40, "min_size": 25, "hole_fill": None, "large_cutoff": cutoff,
            "support_prob": support, "support_raw": None,
            "builder": (lambda cutoff=cutoff, support=support: build_large_component_dilation_mask(
                cutoff, support_kind="prob", support_prob=support)),
        })

for cutoff in [100, 250, 500, 1000]:
    for raw_thr in [0.01, 0.02, 0.05]:
        EXTENDED_CONFIGS.append({
            "config": f"LD{cutoff}_R{raw_thr:.2f}", "method": "large_component_raw_support",
            "threshold": 0.40, "min_size": 25, "hole_fill": None, "large_cutoff": cutoff,
            "support_prob": None, "support_raw": raw_thr,
            "builder": (lambda cutoff=cutoff, raw_thr=raw_thr: build_large_component_dilation_mask(
                cutoff, support_kind="raw", support_raw=raw_thr)),
        })

for prob_thr, raw_thr in [(0.15, 0.01), (0.20, 0.01), (0.20, 0.02)]:
    EXTENDED_CONFIGS.append({
        "config": f"LDCOMB_P{prob_thr:.2f}_R{raw_thr:.2f}", "method": "large_component_combined_support",
        "threshold": 0.40, "min_size": 25, "hole_fill": None, "large_cutoff": 250,
        "support_prob": prob_thr, "support_raw": raw_thr,
        "builder": (lambda prob_thr=prob_thr, raw_thr=raw_thr: build_large_component_dilation_mask(
            250, support_kind="combined", support_prob=prob_thr, support_raw=raw_thr)),
    })

print(f"Prepared {len(EXTENDED_CONFIGS)} extended configurations (baseline handled separately).")

# ---------------------------------------------------------------------------------
# Single-pass metric evaluation (reuses cached target components + persistence lookup)
# ---------------------------------------------------------------------------------
ext_component_rows = []
ext_large_component_rows = []
ext_regime_rows = []
ext_config_rows = []


def evaluate_extended_config(cfg):
    final_mask = cfg["builder"]()
    final_pred = mcfg_raw * final_mask
    predicted_rain = final_pred > 0.01

    dry_bool = ~target_rain
    dry_values = final_pred[dry_bool]
    dry_targets_vals = mcfg_targets[dry_bool]
    dry_mae = float(np.abs(dry_values - dry_targets_vals).mean())
    dry_mean_pred = float(dry_values.mean())
    dry_p95_pred = float(np.percentile(dry_values, 95))
    dry_p99_pred = float(np.percentile(dry_values, 99))
    dry_gt_001 = float((dry_values > 0.01).mean())
    dry_gt_002 = float((dry_values > 0.02).mean())
    dry_gt_005 = float((dry_values > 0.05).mean())

    tp = int((final_mask & target_rain).sum())
    fp = int((final_mask & dry_bool).sum())
    fn = int((~final_mask & target_rain).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    csi = tp / max(tp + fp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    standalone_total = 0
    standalone_pixels_total = 0
    spillover_pixels_total = 0
    standalone_count_per_sample = np.zeros(NUM_SAMPLES, dtype=int)
    ge_counts = {k: [0, 0] for k in GE_THRESHOLDS}
    detection_counts = {b[0]: [0, 0] for b in TRUE_COMPONENT_BINS}
    iou_list, dice_list, area_ratio_list, centroid_list = [], [], [], []
    motion_deltas = []

    for i in range(NUM_SAMPLES):
        pr_labels, pr_count = ndimage.label(predicted_rain[i], structure=STRUCTURE_8)
        for comp_id in range(1, pr_count + 1):
            comp_mask = pr_labels == comp_id
            comp_size = int(comp_mask.sum())
            tp_pixels = int((comp_mask & target_rain[i]).sum())
            fp_pixels = comp_size - tp_pixels
            standalone = tp_pixels == 0
            ys, xs = np.nonzero(comp_mask)
            centroid_y, centroid_x = float(ys.mean()), float(xs.mean())
            h, w = predicted_rain.shape[1], predicted_rain.shape[2]
            edge_distance = float(min(centroid_y, h - 1 - centroid_y, centroid_x, w - 1 - centroid_x))
            ext_component_rows.append({
                "config": cfg["config"], "sample_index": i, "component_id": comp_id, "component_size": comp_size,
                "tp_pixels": tp_pixels, "fp_pixels": fp_pixels, "component_precision": tp_pixels / max(comp_size, 1),
                "standalone": bool(standalone), "centroid_y": centroid_y, "centroid_x": centroid_x,
                "edge_distance": edge_distance,
            })
            if standalone:
                standalone_total += 1
                standalone_pixels_total += fp_pixels
                standalone_count_per_sample[i] += 1
            else:
                spillover_pixels_total += fp_pixels

        fm_labels, _ = ndimage.label(final_mask[i], structure=STRUCTURE_8)
        t_labels = target_labels_cache[i]
        t_count = target_counts_cache[i]
        for comp_id in range(1, t_count + 1):
            comp_mask = t_labels == comp_id
            comp_size = int(comp_mask.sum())
            detected = bool(np.any(final_mask[i] & comp_mask))
            for bin_name, lo, hi in TRUE_COMPONENT_BINS:
                if comp_size >= lo and (hi is None or comp_size <= hi):
                    detection_counts[bin_name][1] += 1
                    detection_counts[bin_name][0] += int(detected)
                    break
            for ge_name, thresh in GE_THRESHOLDS.items():
                if comp_size >= thresh:
                    ge_counts[ge_name][1] += 1
                    ge_counts[ge_name][0] += int(detected)

            if comp_size >= 100:
                overlap_labels = fm_labels[comp_mask]
                overlap_labels = overlap_labels[overlap_labels > 0]
                if overlap_labels.size == 0:
                    matched_id, predicted_size, intersection = None, 0, 0
                else:
                    vals, counts = np.unique(overlap_labels, return_counts=True)
                    matched_id = int(vals[np.argmax(counts)])
                    intersection = int(counts.max())
                    predicted_size = int((fm_labels == matched_id).sum())
                union = comp_size + predicted_size - intersection
                iou = intersection / max(union, 1)
                dice = 2 * intersection / max(comp_size + predicted_size, 1)
                area_ratio = predicted_size / max(comp_size, 1)
                iou_list.append(iou); dice_list.append(dice); area_ratio_list.append(area_ratio)
                forecast_error = np.nan
                if matched_id is not None:
                    tys, txs = np.nonzero(comp_mask)
                    t_cy, t_cx = float(tys.mean()), float(txs.mean())
                    pys, pxs = np.nonzero(fm_labels == matched_id)
                    p_cy, p_cx = float(pys.mean()), float(pxs.mean())
                    forecast_error = float(np.hypot(p_cy - t_cy, p_cx - t_cx))
                    centroid_list.append(forecast_error)
                    persistence_error = persistence_lookup.get((i, comp_id), np.nan)
                    if not (isinstance(persistence_error, float) and np.isnan(persistence_error)):
                        motion_deltas.append(persistence_error - forecast_error)
                ext_large_component_rows.append({
                    "config": cfg["config"], "sample_index": i, "target_component_id": comp_id,
                    "target_size": comp_size, "detected": int(matched_id is not None),
                    "matched_pred_component_id": matched_id, "predicted_size": predicted_size,
                    "intersection_pixels": intersection, "IoU": iou, "Dice": dice,
                    "area_ratio": area_ratio, "centroid_distance": forecast_error,
                })

    total_fp = standalone_pixels_total + spillover_pixels_total
    standalone_fraction = standalone_pixels_total / max(total_fp, 1)
    spillover_fraction = spillover_pixels_total / max(total_fp, 1)
    detection_rates = {b: (detection_counts[b][0] / max(detection_counts[b][1], 1)) for b, _, _ in TRUE_COMPONENT_BINS}
    detection_ge_100 = ge_counts["ge_100"][0] / max(ge_counts["ge_100"][1], 1)
    detection_ge_250 = ge_counts["ge_250"][0] / max(ge_counts["ge_250"][1], 1)
    detection_ge_500 = ge_counts["ge_500"][0] / max(ge_counts["ge_500"][1], 1)

    median_iou = float(np.median(iou_list)) if iou_list else np.nan
    median_dice = float(np.median(dice_list)) if dice_list else np.nan
    median_area_ratio = float(np.median(area_ratio_list)) if area_ratio_list else np.nan
    median_centroid = float(np.median(centroid_list)) if centroid_list else np.nan
    fraction_beats_persistence = float(np.mean([d > 0 for d in motion_deltas])) if motion_deltas else np.nan

    fss_1 = fractions_skill_score(final_mask, target_rain, window=1)
    fss_3 = fractions_skill_score(final_mask, target_rain, window=3)
    fss_5 = fractions_skill_score(final_mask, target_rain, window=5)
    fss_10 = fractions_skill_score(final_mask, target_rain, window=10)

    overall_mae = float(np.abs(final_pred - mcfg_targets).mean())
    rainy_mae = float(np.abs(final_pred[target_rain] - mcfg_targets[target_rain]).mean()) if target_rain.any() else np.nan
    mean_target_rain_intensity = float(mcfg_targets[target_rain].mean()) if target_rain.any() else np.nan
    mean_predicted_rain_intensity = float(final_pred[target_rain].mean()) if target_rain.any() else np.nan
    rain_intensity_ratio = mean_predicted_rain_intensity / max(mean_target_rain_intensity, 1e-12)

    for regime in REGIME_ORDER:
        idx = np.where(frame_regime == regime)[0]
        if len(idx) == 0:
            continue
        r_target_rain = target_rain[idx]
        r_final_mask = final_mask[idx]
        r_final_pred = final_pred[idx]
        r_targets = mcfg_targets[idx]
        tp_r = int((r_final_mask & r_target_rain).sum())
        fp_r = int((r_final_mask & ~r_target_rain).sum())
        fn_r = int((~r_final_mask & r_target_rain).sum())
        p_r = tp_r / max(tp_r + fp_r, 1)
        rc_r = tp_r / max(tp_r + fn_r, 1)
        c_r = tp_r / max(tp_r + fp_r + fn_r, 1)
        f_r = 2 * p_r * rc_r / max(p_r + rc_r, 1e-12)
        rainy_mae_r = float(np.abs(r_final_pred[r_target_rain] - r_targets[r_target_rain]).mean()) if r_target_rain.any() else np.nan
        dry_gt_001_r = float((r_final_pred[~r_target_rain] > 0.01).mean())
        ext_regime_rows.append({
            "config": cfg["config"], "regime": regime, "n_frames": len(idx),
            "precision": p_r, "recall": rc_r, "csi": c_r, "f1": f_r,
            "rainy_mae": rainy_mae_r, "dry_gt_001": dry_gt_001_r,
            "standalone_false_components_per_frame": float(standalone_count_per_sample[idx].mean()),
        })

    return {
        "config": cfg["config"], "method": cfg["method"], "threshold": cfg["threshold"],
        "min_size": cfg["min_size"], "hole_fill": cfg["hole_fill"], "large_cutoff": cfg["large_cutoff"],
        "support_prob": cfg["support_prob"], "support_raw": cfg["support_raw"],
        "precision": precision, "recall": recall, "csi": csi, "f1": f1,
        "dry_mae": dry_mae, "dry_mean_prediction": dry_mean_pred, "dry_p95_prediction": dry_p95_pred,
        "dry_p99_prediction": dry_p99_pred, "dry_gt_001": dry_gt_001, "dry_gt_002": dry_gt_002, "dry_gt_005": dry_gt_005,
        "standalone_false_components_total": standalone_total,
        "standalone_false_components_per_frame": standalone_total / NUM_SAMPLES,
        "standalone_false_pixels_total": standalone_pixels_total,
        "standalone_false_pixels_fraction": standalone_fraction,
        "spillover_false_pixels": spillover_pixels_total, "spillover_fraction_of_all_FP": spillover_fraction,
        **{f"detection_rate_{b}": detection_rates[b] for b in detection_rates},
        "detection_rate_ge_100": detection_ge_100, "detection_rate_ge_250": detection_ge_250, "detection_rate_ge_500": detection_ge_500,
        "large_component_iou_median": median_iou, "large_component_dice_median": median_dice,
        "large_component_area_ratio_median": median_area_ratio, "large_component_centroid_distance_median": median_centroid,
        "fraction_matched_beats_persistence": fraction_beats_persistence,
        "fss_1": fss_1, "fss_3": fss_3, "fss_5": fss_5, "fss_10": fss_10,
        "overall_mae": overall_mae, "rainy_mae": rainy_mae,
        "mean_target_rain_intensity": mean_target_rain_intensity,
        "mean_predicted_rain_intensity": mean_predicted_rain_intensity, "rain_intensity_ratio": rain_intensity_ratio,
    }


for idx_cfg, cfg in enumerate(EXTENDED_CONFIGS):
    ext_config_rows.append(evaluate_extended_config(cfg))
    print(f"[{idx_cfg + 1}/{len(EXTENDED_CONFIGS)}] evaluated {cfg['config']}")

ext_config_df = pd.DataFrame(ext_config_rows)
ext_component_df = pd.DataFrame(ext_component_rows)
ext_large_component_df = pd.DataFrame(ext_large_component_rows)
ext_regime_df = pd.DataFrame(ext_regime_rows)

ext_config_df.to_csv("mask_extended_validation_report.csv", index=False)
ext_component_df.to_csv("mask_extended_component_report.csv", index=False)
ext_large_component_df.to_csv("mask_extended_large_component_report.csv", index=False)
ext_regime_df.to_csv("mask_extended_regime_report.csv", index=False)
print("Saved: mask_extended_validation_report.csv, mask_extended_component_report.csv, "
      "mask_extended_large_component_report.csv, mask_extended_regime_report.csv")

# ---------------------------------------------------------------------------------
# Compact terminal table (baseline A from the previous cell shown for reference)
# ---------------------------------------------------------------------------------
baseline_row = config_df[config_df["config"] == "A_baseline"].iloc[0]
print("\nCOMPACT RANKED TABLE (sorted by CSI, descending) — baseline A shown first for reference")
header = (
    f"{'Config':<20}{'Method':<28}{'Thr':>6}{'MinSz':>6}{'Hole':>6}{'LgCut':>7}{'SupP':>6}{'SupI':>6}"
    f"{'Prec':>7}{'Rec':>7}{'CSI':>7}{'F1':>7}{'Dry>.01':>9}{'DryMAE':>9}{'StandFP':>9}"
    f"{'>=100':>7}{'>=250':>7}{'>=500':>7}{'LgIoU':>8}{'CentErr':>9}{'RainyMAE':>10}{'FSS5':>7}"
)
print(header)


def fmt(row_like, is_baseline=False):
    method = "baseline" if is_baseline else row_like["method"]
    thr = row_like["threshold"] if not is_baseline else row_like["strong_threshold"]
    min_sz = row_like["min_size"] if not is_baseline else row_like["min_component_size"]
    hole = row_like["hole_fill"] if not is_baseline else None
    lg_cut = row_like["large_cutoff"] if not is_baseline else None
    sup_p = row_like["support_prob"] if not is_baseline else row_like["support_threshold"]
    sup_i = row_like["support_raw"] if not is_baseline else None
    dry_mae = row_like["dry_mae"]
    stand_fp = row_like["standalone_false_components_per_frame"]
    det100 = row_like["detection_rate_ge_100"]
    det250 = row_like["detection_rate_ge_250"]
    det500 = row_like["detection_rate_ge_500"]
    lg_iou = row_like["large_component_iou_median"] if not is_baseline else row_like["large_component_iou_mean"]
    cent = row_like["large_component_centroid_distance_median"] if not is_baseline else row_like["large_component_centroid_error_mean"]
    print(
        f"{row_like['config']:<20}{method:<28}{(thr if pd.notna(thr) else -1):>6.3f}"
        f"{(min_sz if pd.notna(min_sz) else -1):>6.0f}{(hole if pd.notna(hole) else -1):>6.0f}"
        f"{(lg_cut if pd.notna(lg_cut) else -1):>7.0f}{(sup_p if pd.notna(sup_p) else -1):>6.2f}"
        f"{(sup_i if pd.notna(sup_i) else -1):>6.2f}{row_like['precision']:>7.4f}{row_like['recall']:>7.4f}"
        f"{row_like['csi']:>7.4f}{row_like['f1']:>7.4f}{row_like['dry_gt_001']:>9.4f}{dry_mae:>9.4f}"
        f"{stand_fp:>9.3f}{det100:>7.3f}{det250:>7.3f}{det500:>7.3f}{lg_iou:>8.4f}{cent:>9.3f}"
        f"{row_like['rainy_mae']:>10.4f}{row_like['fss_5']:>7.4f}"
    )


fmt(baseline_row, is_baseline=True)
for _, r in ext_config_df.sort_values("csi", ascending=False).iterrows():
    fmt(r)

# ---------------------------------------------------------------------------------
# Selection logic vs baseline (hard requirements first, then best CSI among survivors)
# ---------------------------------------------------------------------------------
BASELINE_DRY_GT_001 = baseline_row["dry_gt_001"]
BASELINE_DRY_MAE = baseline_row["dry_mae"]
BASELINE_STANDALONE = baseline_row["standalone_false_components_per_frame"]
BASELINE_LARGE_IOU = baseline_row["large_component_iou_mean"]
BASELINE_CENTROID = baseline_row["large_component_centroid_error_mean"]
BASELINE_DET_100 = baseline_row["detection_rate_ge_100"]
BASELINE_RECALL = baseline_row["recall"]
BASELINE_RAINY_MAE = baseline_row["rainy_mae"]
BASELINE_FSS5 = baseline_row["fss_5"]

hard_ok = (
    (ext_config_df["dry_gt_001"] <= max(BASELINE_DRY_GT_001 * 1.3, 0.01))
    & (ext_config_df["dry_mae"] <= BASELINE_DRY_MAE * 1.3 + 1e-6)
    & (ext_config_df["standalone_false_components_per_frame"] <= BASELINE_STANDALONE * 1.3 + 1e-6)
    & (ext_config_df["large_component_iou_median"] >= BASELINE_LARGE_IOU * 0.95)
    & (ext_config_df["large_component_centroid_distance_median"] <= BASELINE_CENTROID * 1.15)
    & (ext_config_df["detection_rate_ge_100"] >= BASELINE_DET_100 * 0.98)
)
improves_something = (
    (ext_config_df["recall"] > BASELINE_RECALL)
    | (ext_config_df["rainy_mae"] < BASELINE_RAINY_MAE)
    | (ext_config_df["detection_rate_ge_100"] > BASELINE_DET_100)
    | (ext_config_df["large_component_iou_median"] > BASELINE_LARGE_IOU)
    | (ext_config_df["fss_5"] > BASELINE_FSS5)
)
survivors = ext_config_df[hard_ok & improves_something].copy()

if not survivors.empty:
    survivors = survivors.sort_values(["csi", "detection_rate_ge_100", "large_component_iou_median"], ascending=False)
    best_candidate_row = survivors.iloc[0]
    best_candidate_name = best_candidate_row["config"]
else:
    best_candidate_row = None
    best_candidate_name = None

print("\n" + "=" * 70)
print("BASELINE")
print("=" * 70)
print(
    f"threshold=0.40, min_size=25, dilation=0, support=None | "
    f"precision={baseline_row['precision']:.4f}, recall={baseline_row['recall']:.4f}, csi={baseline_row['csi']:.4f}, "
    f"f1={baseline_row['f1']:.4f}, dry_gt_001={baseline_row['dry_gt_001']:.4f}, dry_mae={baseline_row['dry_mae']:.4f}, "
    f"standalone/frame={baseline_row['standalone_false_components_per_frame']:.3f}, "
    f"large IoU={baseline_row['large_component_iou_mean']:.4f}, rainy_mae={baseline_row['rainy_mae']:.4f}"
)

print("\n" + "=" * 70)
print("BEST NON-BASELINE CANDIDATE")
print("=" * 70)
if best_candidate_row is None:
    print("None of the extended configurations passed the hard structural requirements with a genuine improvement.")
else:
    print(
        f"{best_candidate_name} ({best_candidate_row['method']}) | threshold={best_candidate_row['threshold']:.3f}, "
        f"min_size={best_candidate_row['min_size']:.0f}, hole_fill={best_candidate_row['hole_fill']}, "
        f"large_cutoff={best_candidate_row['large_cutoff']}, support_prob={best_candidate_row['support_prob']}, "
        f"support_raw={best_candidate_row['support_raw']}\n"
        f"precision={best_candidate_row['precision']:.4f}, recall={best_candidate_row['recall']:.4f}, "
        f"csi={best_candidate_row['csi']:.4f}, f1={best_candidate_row['f1']:.4f}, "
        f"dry_gt_001={best_candidate_row['dry_gt_001']:.4f}, dry_mae={best_candidate_row['dry_mae']:.4f}, "
        f"standalone/frame={best_candidate_row['standalone_false_components_per_frame']:.3f}, "
        f"large IoU={best_candidate_row['large_component_iou_median']:.4f}, rainy_mae={best_candidate_row['rainy_mae']:.4f}"
    )

print("\n" + "=" * 70)
print("DECISION")
print("=" * 70)
decision = "USE CANDIDATE" if best_candidate_row is not None else "KEEP BASELINE"
print(decision)
print("\nBackground:")
if best_candidate_row is not None:
    print(f"  dry_gt_001 {baseline_row['dry_gt_001']:.4f} -> {best_candidate_row['dry_gt_001']:.4f}, "
          f"dry_mae {baseline_row['dry_mae']:.4f} -> {best_candidate_row['dry_mae']:.4f}")
else:
    print(f"  baseline dry_gt_001={baseline_row['dry_gt_001']:.4f} / dry_mae={baseline_row['dry_mae']:.4f} kept unchanged")
print("Noise:")
if best_candidate_row is not None:
    print(f"  standalone/frame {baseline_row['standalone_false_components_per_frame']:.3f} -> {best_candidate_row['standalone_false_components_per_frame']:.3f}")
else:
    print(f"  baseline standalone/frame={baseline_row['standalone_false_components_per_frame']:.3f} kept unchanged")
print("Large rain:")
if best_candidate_row is not None:
    print(f"  >=100px detection {baseline_row['detection_rate_ge_100']:.3f} -> {best_candidate_row['detection_rate_ge_100']:.3f}")
else:
    print(f"  baseline >=100px detection={baseline_row['detection_rate_ge_100']:.3f} kept unchanged")
print("Spatial quality:")
if best_candidate_row is not None:
    print(f"  large IoU {baseline_row['large_component_iou_mean']:.4f} -> {best_candidate_row['large_component_iou_median']:.4f}")
else:
    print(f"  baseline large IoU={baseline_row['large_component_iou_mean']:.4f} kept unchanged")
print("Intensity:")
if best_candidate_row is not None:
    print(f"  rainy_mae {baseline_row['rainy_mae']:.4f} -> {best_candidate_row['rainy_mae']:.4f}")
else:
    print(f"  baseline rainy_mae={baseline_row['rainy_mae']:.4f} kept unchanged")
print("Trade-off:")
if best_candidate_row is not None:
    print(
        f"  {best_candidate_name} improves on at least one target metric while staying within tolerance on "
        f"background cleanliness, noise rejection, large-component IoU, centroid error and >=100px detection."
    )
else:
    print("  No candidate improved a target metric without materially harming background/noise/large-rain structure; baseline remains Pareto-superior.")

# ---------------------------------------------------------------------------------
# Trade-off plots
# ---------------------------------------------------------------------------------
tfig, taxes = plt.subplots(2, 3, figsize=(19, 11))
plot_specs = [
    (taxes[0, 0], "dry_gt_001", "csi", "dry_gt_001", "CSI"),
    (taxes[0, 1], "precision", "recall", "Precision", "Recall"),
    (taxes[0, 2], "dry_gt_001", "large_component_iou_median", "dry_gt_001", "Large-component IoU (median)"),
    (taxes[1, 0], "dry_gt_001", "rainy_mae", "dry_gt_001", "Rainy MAE"),
    (taxes[1, 1], "recall", "large_component_centroid_distance_median", "Recall", "Centroid error (median)"),
]
for axis, x_col, y_col, x_label, y_label in plot_specs:
    axis.scatter(ext_config_df[x_col], ext_config_df[y_col], s=14)
    axis.scatter([baseline_row["dry_gt_001"] if x_col == "dry_gt_001" else baseline_row.get(x_col, baseline_row["recall"])],
                 [baseline_row.get(y_col if y_col != "large_component_iou_median" else "large_component_iou_mean",
                                   baseline_row.get(y_col if y_col != "large_component_centroid_distance_median" else "large_component_centroid_error_mean", np.nan))],
                 color="red", marker="*", s=160, label="baseline")
    axis.set_xlabel(x_label)
    axis.set_ylabel(y_label)
    axis.grid(True, alpha=0.3)
    axis.legend(fontsize=8)
taxes[1, 2].axis("off")
tfig.suptitle("Extended mask strategy trade-offs (validation set)", fontsize=14)
tfig.tight_layout()
tfig.savefig("mask_extended_tradeoffs.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------------
# Montage: baseline vs best candidate (same validation examples used previously)
# ---------------------------------------------------------------------------------
baseline_mask = config_masks["A_baseline"]
baseline_pred = mcfg_raw * baseline_mask

if best_candidate_name is not None:
    best_cfg = next(c for c in EXTENDED_CONFIGS if c["config"] == best_candidate_name)
    candidate_mask = best_cfg["builder"]()
else:
    candidate_mask = baseline_mask
candidate_pred = mcfg_raw * candidate_mask

montage_groups = [
    ("Dry/near-dry", np.where(np.isin(frame_regime, ["completely_dry", "very_light"]))[0]),
    ("Light", np.where(frame_regime == "light")[0]),
    ("Moderate", np.where(frame_regime == "moderate")[0]),
    ("Heavy", np.where(frame_regime == "heavy")[0]),
]
montage_indices, montage_labels = [], []
for group_name, idx_pool in montage_groups:
    chosen = idx_pool[:5]
    montage_indices.extend(chosen.tolist())
    montage_labels.extend([f"{group_name} #{n + 1}" for n in range(len(chosen))])

montage_columns = ["Target", "Raw prediction", "Rain probability", "Baseline mask", "Baseline prediction", "Candidate mask", "Candidate prediction"]
mfig, maxes = plt.subplots(
    len(montage_indices), len(montage_columns),
    figsize=(3.4 * len(montage_columns), 2.8 * len(montage_indices)), squeeze=False,
)
for row_i, (sample_i, row_label) in enumerate(zip(montage_indices, montage_labels)):
    vmax = max(float(mcfg_targets[sample_i].max()), float(mcfg_raw[sample_i].max()), 0.5)
    row_panels = [
        (mcfg_targets[sample_i], "viridis", 0.0, vmax),
        (mcfg_raw[sample_i], "viridis", 0.0, vmax),
        (mcfg_prob[sample_i], "magma", 0.0, 1.0),
        (baseline_mask[sample_i].astype(float), "gray", 0.0, 1.0),
        (baseline_pred[sample_i], "viridis", 0.0, vmax),
        (candidate_mask[sample_i].astype(float), "gray", 0.0, 1.0),
        (candidate_pred[sample_i], "viridis", 0.0, vmax),
    ]
    for col_i, (image, cmap, vmin, vm) in enumerate(row_panels):
        axis = maxes[row_i, col_i]
        axis.imshow(image, cmap=cmap, vmin=vmin, vmax=vm)
        axis.set_xticks([]); axis.set_yticks([])
        if row_i == 0:
            axis.set_title(montage_columns[col_i], fontsize=10)
        if col_i == 0:
            axis.set_ylabel(row_label, fontsize=9)
mfig.suptitle(f"Baseline vs best extended candidate ({best_candidate_name if best_candidate_name else 'none (baseline kept)'})", fontsize=14)
mfig.tight_layout()
mfig.savefig("mask_extended_baseline_vs_best.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Calibration/strategy diagnostics below evaluate +5 minutes only.
# Use plot_validation_examples() above for paired +5/+10 forecast displays.

# =====================================================================================
# BUG FIX: A_baseline (from the first evaluation cell) and H4 (from the extended sweep)
# are mathematically identical configurations (threshold=0.40, min_size=25, no
# dilation/support) but were reported with different large-component IoU/centroid
# numbers. Root cause: the first cell reported MEAN IoU / MEAN centroid error, while
# the extended sweep reports MEDIAN IoU / MEDIAN centroid error for every config -
# comparing a mean to a median on a skewed distribution looks like a discrepancy but
# is actually an apples-to-oranges comparison, not a computational bug in the masks.
# Verify this explicitly, then evaluate A_baseline through the EXACT same function
# used for every other configuration (no separately cached/precomputed baseline).
# =====================================================================================
mask_A_baseline = build_hard_threshold_mask(0.40, 25)
h4_cfg = next(c for c in EXTENDED_CONFIGS if c["config"] == "H4")
mask_H4 = h4_cfg["builder"]()

assert np.array_equal(mask_A_baseline, mask_H4), "A_baseline and H4 masks differ - real bug, not just a stats mismatch."
print("Confirmed: A_baseline and H4 produce bit-identical final masks.")

baseline_cfg = {
    "config": "A_baseline", "method": "hard_threshold", "threshold": 0.40, "min_size": 25,
    "hole_fill": None, "large_cutoff": None, "support_prob": None, "support_raw": None,
    "builder": (lambda: build_hard_threshold_mask(0.40, 25)),
}
baseline_row_consistent = evaluate_extended_config(baseline_cfg)
h4_row_existing = ext_config_df[ext_config_df["config"] == "H4"].iloc[0].to_dict()

metric_keys_to_check = [
    "precision", "recall", "csi", "f1", "dry_mae", "dry_gt_001", "dry_gt_002", "dry_gt_005",
    "standalone_false_components_per_frame", "detection_rate_ge_100", "detection_rate_ge_250",
    "detection_rate_ge_500", "large_component_iou_median", "large_component_dice_median",
    "large_component_area_ratio_median", "large_component_centroid_distance_median",
    "fraction_matched_beats_persistence", "fss_1", "fss_3", "fss_5", "fss_10", "rainy_mae",
]
mismatches = []
for key in metric_keys_to_check:
    a, b = baseline_row_consistent[key], h4_row_existing[key]
    both_nan = isinstance(a, float) and isinstance(b, float) and np.isnan(a) and np.isnan(b)
    if not both_nan and a != b:
        mismatches.append((key, a, b))

if mismatches:
    print("MISMATCHES FOUND (real bug):")
    for key, a, b in mismatches:
        print(f"  {key}: A_baseline={a}, H4={b}")
    raise AssertionError("A_baseline and H4 metrics differ when computed through the identical evaluation function.")
else:
    print("Confirmed: every derived metric (precision/recall/CSI/F1, dry stats, standalone/frame, ")
    print("detection rates, large-component IoU/Dice/area_ratio/centroid, persistence, FSS) is IDENTICAL")
    print("between A_baseline and H4 once both are run through evaluate_extended_config().")
    print("\nThe earlier apparent large-IoU/centroid discrepancy (0.4972 vs 0.5310, 6.916 vs 3.560) was")
    print("caused by comparing MEAN (first cell) against MEDIAN (extended sweep) of a skewed distribution,")
    print("not by a computational error in the mask or metric logic.")

# Replace the ext_config_df row for A_baseline (if present) with the consistent one, else append.
ext_config_df = ext_config_df[ext_config_df["config"] != "A_baseline"]
ext_config_df = pd.concat([ext_config_df, pd.DataFrame([baseline_row_consistent])], ignore_index=True)

# ---------------------------------------------------------------------------------
# Final fine threshold sweep (min_size=25, no dilation, no support) using the SAME
# evaluation function. Reuse H5/H6/H7 (0.425/0.45/0.50) already computed consistently;
# add the remaining thresholds requested.
# ---------------------------------------------------------------------------------
FINE_THRESHOLDS_NEW = [0.475, 0.525, 0.55, 0.575, 0.60]
for thr in FINE_THRESHOLDS_NEW:
    cname = f"FINE_{thr:.3f}"
    cfg = {
        "config": cname, "method": "hard_threshold", "threshold": thr, "min_size": 25,
        "hole_fill": None, "large_cutoff": None, "support_prob": None, "support_raw": None,
        "builder": (lambda thr=thr: build_hard_threshold_mask(thr, 25)),
    }
    ext_config_df = ext_config_df[ext_config_df["config"] != cname]
    ext_config_df = pd.concat([ext_config_df, pd.DataFrame([evaluate_extended_config(cfg)])], ignore_index=True)
    print(f"Evaluated {cname} (threshold={thr:.3f})")

ext_config_df.to_csv("mask_extended_validation_report.csv", index=False)
ext_component_df = pd.DataFrame(ext_component_rows)
ext_large_component_df = pd.DataFrame(ext_large_component_rows)
ext_regime_df = pd.DataFrame(ext_regime_rows)
ext_component_df.to_csv("mask_extended_component_report.csv", index=False)
ext_large_component_df.to_csv("mask_extended_large_component_report.csv", index=False)
ext_regime_df.to_csv("mask_extended_regime_report.csv", index=False)
print("Re-saved extended CSVs including the consistent A_baseline row and the new fine-threshold configs.")

# ---------------------------------------------------------------------------------
# Final fully-consistent ranked table restricted to the hard-threshold family
# (A_baseline + H1-H7 + new fine thresholds), all evaluated through the same function.
# ---------------------------------------------------------------------------------
threshold_family = ext_config_df[ext_config_df["method"].isin(["hard_threshold"])].copy()
threshold_family = threshold_family.sort_values("threshold")
print("\nHARD-THRESHOLD FAMILY (fully consistent metrics, min_size=25, no dilation/support)")
print(f"{'Config':<14}{'Thr':>7}{'Prec':>8}{'Rec':>8}{'CSI':>8}{'F1':>8}{'Dry>.01':>9}{'DryMAE':>9}"
      f"{'StandFP':>9}{'>=100':>7}{'>=250':>7}{'>=500':>7}{'LgIoU':>8}{'CentErr':>9}{'RainyMAE':>10}{'FSS5':>7}")
for _, r in threshold_family.iterrows():
    print(
        f"{r['config']:<14}{r['threshold']:>7.3f}{r['precision']:>8.4f}{r['recall']:>8.4f}{r['csi']:>8.4f}{r['f1']:>8.4f}"
        f"{r['dry_gt_001']:>9.4f}{r['dry_mae']:>9.4f}{r['standalone_false_components_per_frame']:>9.3f}"
        f"{r['detection_rate_ge_100']:>7.3f}{r['detection_rate_ge_250']:>7.3f}{r['detection_rate_ge_500']:>7.3f}"
        f"{r['large_component_iou_median']:>8.4f}{r['large_component_centroid_distance_median']:>9.3f}"
        f"{r['rainy_mae']:>10.4f}{r['fss_5']:>7.4f}"
    )

best_threshold_row = threshold_family.sort_values("csi", ascending=False).iloc[0]
print(f"\nBest by CSI within the hard-threshold family: {best_threshold_row['config']} "
      f"(threshold={best_threshold_row['threshold']:.3f}, CSI={best_threshold_row['csi']:.4f})")

In [ ]:
# HELD-OUT TEST: frozen p=0.40, minimum size=40 for both leads
# Uses model2 and the existing chronological test_loader; no threshold tuning.
from scipy import ndimage

selected_threshold_test = 0.40
selected_component_size_test = 40

def horizon_clean_mask(probability, threshold, minimum_size):
    cleaned = []
    for sample in probability.numpy():
        components, _ = ndimage.label(sample >= threshold, structure=np.ones((3, 3)))
        keep = np.bincount(components.ravel()) >= minimum_size
        keep[0] = False
        cleaned.append(torch.from_numpy(keep[components]))
    return torch.stack(cleaned)


def horizon_accumulate(stats, mask, prediction, target):
    rain = target > 0.01
    dry = ~rain
    stats['tp'] += (mask & rain).sum().item()
    stats['fp'] += (mask & dry).sum().item()
    stats['fn'] += (~mask & rain).sum().item()
    stats['rain_n'] += rain.sum().item()
    stats['dry_n'] += dry.sum().item()
    error = (prediction - target).abs()
    stats['rain_error'] += error[rain].sum().item()
    stats['dry_error'] += error[dry].sum().item()
    stats['dry_wet'] += (prediction[dry] > 0.01).sum().item()


def horizon_ratio(numerator, denominator):
    return numerator / denominator if denominator else float('nan')


horizon_totals = {
    (lead, method): dict.fromkeys(
        ['tp', 'fp', 'fn', 'rain_n', 'dry_n', 'rain_error', 'dry_error', 'dry_wet'], 0
    )
    for lead in (5, 10)
    for method in ('Model mask', 'Model intensity > .01', 'Persistence')
}
model2.eval()
heldout_sample_count = 0
with torch.no_grad():
    for test_batch, (metric_inputs, metric_targets) in enumerate(test_loader, 1):
        if metric_targets.ndim != 4 or metric_targets.shape[1] != 2:
            raise ValueError('Expected targets [batch, 2, height, width].')
        metric_inputs = metric_inputs.to(device)
        raw5, _, _, logits5 = model2(metric_inputs, return_logits=True)
        # Raw, ungated +5 goes into the +10 rollout, exactly as in training.
        raw10, _, _, logits10 = model2(
            build_rollout_inputs(metric_inputs, raw5), return_logits=True
        )
        persistence = metric_inputs[:, -1, 0].cpu().clamp_min(0)
        for index, (lead, raw, logits) in enumerate(((5, raw5, logits5), (10, raw10, logits10))):
            target = metric_targets[:, index].cpu()
            probability = logits.sigmoid().squeeze(1).cpu()
            raw = raw.squeeze(1).cpu().clamp_min(0)
            mask = horizon_clean_mask(probability, selected_threshold_test, selected_component_size_test)
            gated = raw * mask
            horizon_accumulate(horizon_totals[lead, 'Model mask'], mask, gated, target)
            horizon_accumulate(horizon_totals[lead, 'Model intensity > .01'], gated > 0.01, gated, target)
            horizon_accumulate(horizon_totals[lead, 'Persistence'], persistence > 0.01, persistence, target)
        heldout_sample_count += len(metric_targets)
        if test_batch % 5 == 0:
            print(f'Test progress: {test_batch}/{len(test_loader)} batches', flush=True)

if not heldout_sample_count:
    raise RuntimeError('Test loader is empty.')

heldout_test_results = []
for (lead, method), stats in horizon_totals.items():
    tp, fp, fn = (stats[key] for key in ('tp', 'fp', 'fn'))
    heldout_test_results.append({
        'lead_min': lead, 'method': method,
        'precision': horizon_ratio(tp, tp + fp),
        'recall': horizon_ratio(tp, tp + fn),
        'CSI': horizon_ratio(tp, tp + fp + fn),
        'F1': horizon_ratio(2 * tp, 2 * tp + fp + fn),
        'rainy_MAE': horizon_ratio(stats['rain_error'], stats['rain_n']),
        'dry_MAE': horizon_ratio(stats['dry_error'], stats['dry_n']),
        'dry_gt_001': horizon_ratio(stats['dry_wet'], stats['dry_n']),
    })
print(f'HELD-OUT TEST: {heldout_sample_count} samples | loaded best checkpoint (model2)')
print(f'Both leads: probability >= {selected_threshold_test:.2f}, component size >= {selected_component_size_test}')
print('Pixel-pooled metrics; MAE uses normalized radar intensity, not mm/hr.')
print('Model mask scores the cleaned probability mask; Model intensity scores the final gated radar.')
print('Persistence repeats the last preprocessed input radar at both horizons.')
print(pd.DataFrame(heldout_test_results).to_string(index=False, float_format=lambda x: f'{x:.4f}'))


## Held-out test result — 11 September 2026

Executed externally with `scripts/run_heldout_test.py`, using the saved best checkpoint and normalization, frozen probability threshold 0.40 and minimum component size 40. Original dataset count verified: 4,330 samples; final 433 evaluated. Persistence uses the preprocessed last input radar. No calibration or training was run.

```text
HELD-OUT TEST: 433 samples | loaded best checkpoint (model2)
Both leads: probability >= 0.40, component size >= 40
Pixel-pooled metrics; MAE uses normalized radar intensity, not mm/hr.
Model mask scores the cleaned probability mask; Model intensity scores the final gated radar.
Persistence repeats the last preprocessed input radar at both horizons.
 lead_min                method  precision  recall    CSI     F1  rainy_MAE  dry_MAE  dry_gt_001
        5            Model mask     0.7370  0.5682 0.4725 0.6417     0.1262   0.0003      0.0021
        5 Model intensity > .01     0.7372  0.5679 0.4722 0.6415     0.1262   0.0003      0.0021
        5           Persistence     0.5188  0.5174 0.3496 0.5181     0.1488   0.0008      0.0049
       10            Model mask     0.6870  0.5097 0.4137 0.5852     0.1505   0.0004      0.0024
       10 Model intensity > .01     0.6870  0.5096 0.4136 0.5851     0.1505   0.0004      0.0024
       10           Persistence     0.4323  0.4337 0.2763 0.4330     0.1872   0.0010      0.0058
```

Both horizons outperform persistence, but test recall is lower than validation. These pixel metrics do not directly measure notification accuracy. Do not tune settings against these test results. Checkpoint/normalization hashes are recorded in `heldout_test_metadata.json`.


In [ ]:
# SOURCE / REPLAY REPORTS — no retraining and no target relabeling
# The current notebook/model remains on legacy decoding. For a NEW training
# experiment, explicitly construct a dataset with decoder_version=SOURCE from
# data_processing.radar_codec, fit its normalization and save model metadata.
from pathlib import Path
import json
import pandas as pd
replay_dir = project_root / "reports" / "notification_replay"
if (replay_dir / "summary.json").exists():
    replay_summary = json.loads((replay_dir / "summary.json").read_text())
    print("Continuous validation replay (ideal delivery, synthetic grid unless configured):")
    print(pd.DataFrame(replay_summary["leads"]).T.to_string())
    print("Mask settings:", replay_summary["mask_threshold"], replay_summary["min_size"])
    print("Tiny echoes remain in all targets; size-specific recall is diagnostic only.")
    for lead, groups in replay_summary["region_metrics"].items():
        print(f"+{lead} target-region metrics:")
        print(pd.DataFrame({k:v for k,v in groups.items() if k != "all_pixels"}).T.to_string())
        print("Original full-radar counts:", groups["all_pixels"])
else:
    print("Run scripts/replay_validation_notifications.py from the project root first.")


## Completed continuous validation replay

The source decoder and versioned caches are implemented; the existing model remains on legacy input values. Actual-radar captions use the official source categories.

Offline replay covered 3,262 usable five-minute timestamps at 25 synthetic locations, with 1,844 missing-input timestamps treated as gaps. At the production 0.55/25 mask settings, later radar contradicted 25/108 scored +5 predicted-clear alerts and 34/136 hypothetical +10 alerts. Tiny echoes are retained, so these are not independent live-user incident rates.

See `reports/notification_replay/report.md` and run the report cell above to inspect counts by region size. No notification rules or training losses were changed.
